# Historical reproduction: MARBERT vs AraBERT (audit item 1)

**Paper:** Abdul-Mageed, Elmadany & Nagoudi (2021), *ARBERT & MARBERT*, ACL-IJCNLP.
**Protocol (as in the paper):** train on the **full** training set, score the test set after every epoch,
report the **best epoch on test** with the published metric. This is deliberately the historical
(best-on-test) protocol and must be labelled as such in the report.

| Dataset | Metric | Published MARBERT / AraBERT |
|---|---|---|
| ArSarcasm-Sent | F1-PN | 71.50 / 63.50 |
| ASTD (trained on CAMelSent) | F1-PN | 78.00 / 72.00 |
| AJGT | Accuracy | 96.11 / 92.22 |

### How to use
1. `Runtime → Change runtime type → GPU` (T4 is enough).
2. Put Member 1's audited data in Google Drive (see **Section 3** for the expected layout).
3. Edit **Section 2 (Config)** if your folder differs, then `Runtime → Run all`.
4. Runs are **resumable**: if Colab disconnects, just run all cells again. Finished runs are skipped,
   interrupted runs continue from the last saved epoch.
5. When all 18 runs are done, **Section 9** builds the published-vs-ours table and the verdicts.

## 1. Environment: GPU check, Google Drive, package versions

In [2]:
import os, sys, json, subprocess, platform, time
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

import shutil as _sh
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip() if _sh.which("nvidia-smi") else "no NVIDIA GPU found"
if IN_COLAB and gpu_info == "no NVIDIA GPU found":
    raise RuntimeError("No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again.")
print("GPU:", gpu_info)

Mounted at /content/drive
GPU: NVIDIA L4, 580.82.07, 23034 MiB


In [4]:
# Key packages are locked on the first session and re-installed with the SAME versions in later sessions,
# so all 18 runs use identical software even if Colab updates its defaults in between.
PROJECT_DIR = "/content/drive/MyDrive/marbert_repro" if IN_COLAB else os.path.abspath("marbert_repro")
os.makedirs(PROJECT_DIR, exist_ok=True)
LOCK_FILE = f"{PROJECT_DIR}/env_lock.json"
LOCKED_PKGS = ["transformers", "tokenizers", "huggingface_hub", "scikit-learn", "safetensors"]

from importlib.metadata import version as _v
if os.path.exists(LOCK_FILE):
    lock = json.load(open(LOCK_FILE))
    need = [f"{p}=={lock[p]}" for p in LOCKED_PKGS if p in lock and _v(p) != lock[p]]
    if need:
        print("Installing locked versions:", need)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
        raise SystemExit(">>> Locked versions installed. Do Runtime -> Restart session, then Run all again.")
    import torch
    if torch.__version__ != lock.get("torch"):
        print(f"WARNING: torch {torch.__version__} differs from locked {lock.get('torch')} - note it in departures.md")
else:
    import torch
    lock = {p: _v(p) for p in LOCKED_PKGS}
    lock.update({"torch": torch.__version__, "python": platform.python_version(), "gpu_first_session": gpu_info})
    json.dump(lock, open(LOCK_FILE, "w"), indent=2)
    print("Created", LOCK_FILE)
print(json.dumps(lock, indent=2))

{
  "transformers": "5.22.1",
  "tokenizers": "0.23.3",
  "huggingface_hub": "1.33.0",
  "scikit-learn": "1.6.1",
  "safetensors": "0.8.0",
  "torch": "2.14.1+cu130",
  "python": "3.13.16",
  "gpu_first_session": "Tesla T4, 580.82.07, 15360 MiB"
}


## 2. Config
Everything the paper specifies is here. Change only paths/seeds unless you log a departure.

In [6]:
DATA_DIR  = f"{PROJECT_DIR}/data"                     # Member 1's audited files
OUT_DIR   = f"{PROJECT_DIR}/outputs/historical"        # predictions + metrics (keep forever)
CKPT_DIR  = f"{PROJECT_DIR}/checkpoints/historical"    # resume checkpoints (deleted when a run finishes)

MODELS = {
    "marbert": "UBC-NLP/MARBERT",                 # original MARBERT, NOT MARBERTv2
    "arabert": "aubmindlab/bert-base-arabertv01", # AraBERT not named in [1]; our fixed choice (logged)
}

DATASETS = {
    # labels: order = class ids. metric: historical metric used to pick the best epoch and compare.
    "arsarcasm": {"labels": ["negative", "neutral", "positive"], "metric": "f1_pn",
                  "published": {"marbert": 71.50, "arabert": 63.50}},
    "astd":      {"labels": ["negative", "neutral", "positive"], "metric": "f1_pn",
                  "published": {"marbert": 78.00, "arabert": 72.00}},
    "ajgt":      {"labels": ["negative", "positive"], "metric": "accuracy",
                  "published": {"marbert": 96.11, "arabert": 92.22}},
}

# Hyperparameters reported in [1]
HP = dict(lr=2e-6, batch_size=32, max_len=256, epochs=25, weight_decay=0.0,
          optimizer="Adam (torch.optim.Adam)", scheduler="constant (unreported in [1]; assumption)", warmup=0)

SEEDS = [42, 43, 44]
RUN_ORDER = ["astd", "arsarcasm", "ajgt"]   # longest first

# Implementation choices NOT specified in [1] - each one is listed in departures.md
USE_AMP = True            # fp16 mixed precision (faster on T4). Set False for full fp32.
PREPROC = dict(replace_user=True, replace_url=True)  # @mentions -> USER, links -> URL (verify vs MARBERT model card)
TEXT_COL, LABEL_COL = "text", "label"
# Resume checkpoints (~2 GB each for MARBERT incl. optimizer; writing one to Drive takes ~20-60 s).
# Save every N epochs per dataset; 0 = never (short runs simply restart if Colab disconnects).
CKPT_EVERY = {"astd": 5, "arsarcasm": 0, "ajgt": 0}

## 3. Data
Expected layout in Drive (CSV, UTF-8, columns `text` and `label`):
```
marbert_repro/data/
  arsarcasm/train.csv   arsarcasm/test.csv
  astd/train.csv        astd/test.csv      # train = merged CAMelSent (train+dev of ArSAS, ASTD, SemEval, ArSenTD-LEV); test = ASTD test
  ajgt/train.csv        ajgt/test.csv      # split agreed with Member 1 (no official split - log it)
```
Labels may be `POS/NEG/NEU`, `positive/negative/neutral`, `Positive/Negative`, etc.; they are normalised below.
Any other label value stops the notebook so it can be fixed in the data audit, not silently dropped.

In [7]:
import re, random, hashlib
import numpy as np, pandas as pd

LABEL_ALIASES = {"pos": "positive", "positive": "positive", "neg": "negative", "negative": "negative",
                 "neu": "neutral", "neut": "neutral", "neutral": "neutral"}

URL_RE  = re.compile(r"(https?://\S+|www\.\S+)")
USER_RE = re.compile(r"@\w+")
def preprocess(t: str) -> str:
    t = str(t)
    if PREPROC["replace_url"]:  t = URL_RE.sub("URL", t)
    if PREPROC["replace_user"]: t = USER_RE.sub("USER", t)
    return re.sub(r"\s+", " ", t).strip()

def load_split(dataset: str, split: str) -> pd.DataFrame:
    path = f"{DATA_DIR}/{dataset}/{split}.csv"
    df = pd.read_csv(path)
    missing = {TEXT_COL, LABEL_COL} - set(df.columns)
    assert not missing, f"{path} is missing columns {missing}"
    df = df[[TEXT_COL, LABEL_COL]].rename(columns={TEXT_COL: "text", LABEL_COL: "label"})
    raw = df["label"].astype(str).str.strip().str.lower()
    df["label"] = raw.map(LABEL_ALIASES)
    bad = sorted(raw[df["label"].isna()].unique())
    assert not bad, f"{path}: unknown labels {bad} - fix in the data audit / LABEL_ALIASES"
    labels = DATASETS[dataset]["labels"]
    extra = set(df["label"]) - set(labels)
    assert not extra, f"{path}: labels {extra} not expected for {dataset}"
    df["y"] = df["label"].map({l: i for i, l in enumerate(labels)})
    df["text"] = df["text"].map(preprocess)
    return df.reset_index(drop=True)

def file_md5(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()

# Sanity check: counts per split and label (copy into the report / compare with Member 1's audit)
for ds in DATASETS:
    for sp in ["train", "test"]:
        p = f"{DATA_DIR}/{ds}/{sp}.csv"
        if os.path.exists(p):
            d = load_split(ds, sp)
            print(f"{ds:10s} {sp:5s} n={len(d):6d}", d["label"].value_counts().to_dict())
        else:
            print(f"{ds:10s} {sp:5s} MISSING -> {p}")

arsarcasm  train n=  8437 {'neutral': 4262, 'negative': 2813, 'positive': 1362}
arsarcasm  test  n=  2110 {'neutral': 1078, 'negative': 716, 'positive': 316}
astd       train n= 27394 {'negative': 11241, 'neutral': 9483, 'positive': 6670}
astd       test  n=   661 {'negative': 336, 'neutral': 166, 'positive': 159}
ajgt       train n=  1440 {'negative': 720, 'positive': 720}
ajgt       test  n=   360 {'positive': 180, 'negative': 180}


## 4. Metrics (+ unit test)
* **F1-PN** = mean of positive-class F1 and negative-class F1, computed on the **full 3-class test set**
  (neutral examples kept, neutral F1 not averaged in).
* Every epoch also logs accuracy, macro-F1 and per-class precision / recall / F1 / support.

In [5]:
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix

def compute_metrics(y_true, y_pred, labels):
    ids = list(range(len(labels)))
    m = {"accuracy": 100 * accuracy_score(y_true, y_pred),
         "macro_f1": 100 * f1_score(y_true, y_pred, labels=ids, average="macro", zero_division=0)}
    if "positive" in labels and "negative" in labels:
        pn = [labels.index("positive"), labels.index("negative")]
        m["f1_pn"] = 100 * f1_score(y_true, y_pred, labels=pn, average="macro", zero_division=0)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=ids, zero_division=0)
    m["per_class"] = {l: {"precision": 100 * p[i], "recall": 100 * r[i], "f1": 100 * f[i], "support": int(s[i])}
                      for i, l in enumerate(labels)}
    m["confusion_matrix"] = confusion_matrix(y_true, y_pred, labels=ids).tolist()
    return m

# Unit test (hand-computed): pos F1 = .5, neg F1 = 1, neu F1 = .5  ->  F1-PN = 75, macro-F1 = 66.67, acc = 66.67
_L = ["negative", "neutral", "positive"]; N, U, P = 0, 1, 2
_m = compute_metrics([P, P, N, N, U, U], [P, U, N, N, U, P], _L)
assert abs(_m["f1_pn"] - 75.0) < 1e-6 and abs(_m["macro_f1"] - 200/3) < 1e-6 and abs(_m["accuracy"] - 400/6) < 1e-6
print("metric unit test passed")

metric unit test passed


## 5. Seeding and model revisions
Model commit hashes are resolved once and stored, so every run uses the identical checkpoint.

In [6]:
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
import transformers

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

REV_FILE = f"{PROJECT_DIR}/model_revisions.json"
def get_revisions():
    revs = json.load(open(REV_FILE)) if os.path.exists(REV_FILE) else {}
    changed = False
    for k, mid in MODELS.items():
        if k not in revs:
            if os.path.isdir(mid):
                revs[k] = None                       # local folder (testing)
            else:
                from huggingface_hub import HfApi
                revs[k] = HfApi().model_info(mid).sha
            changed = True
    if changed: json.dump(revs, open(REV_FILE, "w"), indent=2)
    return revs

REVISIONS = get_revisions()
print(json.dumps(REVISIONS, indent=2))

device: cuda
{
  "marbert": "88e1fa192dd723cf0b3563500aec46209762eb22",
  "arabert": "01f6b7d2d17afcd0f79498f26f47dfc6207b56ea"
}


## 6. One training run (resumable)
One run = one dataset × one model × one seed. After every epoch it writes, to Drive:
`metrics_per_epoch.jsonl`, `preds/epoch_XX_test_preds.csv` (gold, prediction, class probabilities),
and a resume checkpoint. A `DONE` file marks a finished run.

In [7]:
import shutil
from torch.utils.data import DataLoader

class EncDataset(torch.utils.data.Dataset):
    def __init__(self, enc, y): self.enc, self.y = enc, y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}; item["labels"] = int(self.y[i]); return item

def _autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and DEVICE == "cuda"))

@torch.no_grad()
def predict(model, loader):
    model.eval(); probs = []
    for b in loader:
        b = {k: v.to(DEVICE) for k, v in b.items() if k != "labels"}
        with _autocast():
            logits = model(**b).logits
        probs.append(torch.softmax(logits.float(), -1).cpu())
    return torch.cat(probs).numpy()

def train_one_run(dataset, model_key, seed, out_root=None, ckpt_root=None, max_train=None, epochs=None):
    out_root, ckpt_root = out_root or OUT_DIR, ckpt_root or CKPT_DIR
    epochs = epochs or HP["epochs"]
    labels = DATASETS[dataset]["labels"]
    run_dir  = f"{out_root}/{dataset}/{model_key}/seed{seed}"
    ckpt_path = f"{ckpt_root}/{dataset}/{model_key}/seed{seed}/last.pt"
    if os.path.exists(f"{run_dir}/DONE"):
        print(f"[skip] {dataset}/{model_key}/seed{seed} already done"); return
    os.makedirs(f"{run_dir}/preds", exist_ok=True); os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

    set_seed(seed)
    train_df, test_df = load_split(dataset, "train"), load_split(dataset, "test")
    if max_train: train_df = train_df.sample(n=min(max_train, len(train_df)), random_state=seed).reset_index(drop=True)

    mid, rev = MODELS[model_key], REVISIONS[model_key]
    tok = AutoTokenizer.from_pretrained(mid, revision=rev)
    model = AutoModelForSequenceClassification.from_pretrained(mid, revision=rev, num_labels=len(labels)).to(DEVICE)
    enc = lambda df: tok(df["text"].tolist(), truncation=True, max_length=HP["max_len"])
    train_ds, test_ds = EncDataset(enc(train_df), train_df["y"].values), EncDataset(enc(test_df), test_df["y"].values)
    collate = DataCollatorWithPadding(tok)   # dynamic padding: pad to longest tweet in batch (logged)
    test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, collate_fn=collate)

    opt = torch.optim.Adam(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    start_epoch = 1
    metrics_file = f"{run_dir}/metrics_per_epoch.jsonl"

    if os.path.exists(ckpt_path):                                   # ---- resume
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        torch.set_rng_state(ck["rng_cpu"])
        if DEVICE == "cuda" and ck.get("rng_cuda") is not None: torch.cuda.set_rng_state_all(ck["rng_cuda"])
        start_epoch = ck["epoch"] + 1
        if os.path.exists(metrics_file):                            # drop epochs logged after the checkpoint
            keep = [l for l in open(metrics_file) if json.loads(l)["epoch"] <= ck["epoch"]]
            open(metrics_file, "w").writelines(keep)
        print(f"[resume] {dataset}/{model_key}/seed{seed} from epoch {start_epoch}")
    else:
        if os.path.exists(metrics_file): os.remove(metrics_file)

    info = dict(protocol="historical: full training set, best epoch selected on TEST (as in [1])",
                dataset=dataset, model_key=model_key, model_id=mid, model_revision=rev, seed=seed,
                labels=labels, n_train=len(train_df), n_test=len(test_df),
                train_md5=file_md5(f"{DATA_DIR}/{dataset}/train.csv"), test_md5=file_md5(f"{DATA_DIR}/{dataset}/test.csv"),
                hp=HP, epochs=epochs, use_amp=USE_AMP, preprocessing=PREPROC, dynamic_padding=True, max_train=max_train,
                gpu=gpu_info, torch=torch.__version__, transformers=transformers.__version__,
                started=time.strftime("%Y-%m-%d %H:%M:%S"))
    json.dump(info, open(f"{run_dir}/run_info.json", "w"), indent=2, ensure_ascii=False)

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time(); model.train()
        g = torch.Generator(); g.manual_seed(seed * 1000 + epoch)   # data order depends only on (seed, epoch) -> safe resume
        loader = DataLoader(train_ds, batch_size=HP["batch_size"], shuffle=True, generator=g, collate_fn=collate)
        tot, n = 0.0, 0
        for b in loader:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            opt.zero_grad(set_to_none=True)
            with _autocast():
                loss = model(**b).loss
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * b["labels"].size(0); n += b["labels"].size(0)

        probs = predict(model, test_loader); pred = probs.argmax(1)
        m = compute_metrics(test_df["y"].values, pred, labels)
        m.update(epoch=epoch, train_loss=tot / n, seconds=round(time.time() - t0, 1))
        with open(metrics_file, "a") as f: f.write(json.dumps(m) + "\n")
        pdf = pd.DataFrame({"idx": np.arange(len(test_df)), "gold": test_df["y"].values, "pred": pred})
        for i, l in enumerate(labels): pdf[f"p_{l}"] = probs[:, i].round(6)
        pdf.to_csv(f"{run_dir}/preds/epoch_{epoch:02d}_test_preds.csv", index=False)
        key = DATASETS[dataset]["metric"]
        print(f"{dataset}/{model_key}/seed{seed} ep{epoch:02d} loss={tot/n:.4f} {key}={m[key]:.2f} "
              f"macroF1={m['macro_f1']:.2f} ({m['seconds']}s)")

        ce = CKPT_EVERY.get(dataset, 1) if isinstance(CKPT_EVERY, dict) else CKPT_EVERY
        if ce and epoch % ce == 0 and epoch < epochs:                # ---- checkpoint (atomic write)
            tmp = ckpt_path + ".tmp"
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "epoch": epoch, "rng_cpu": torch.get_rng_state(),
                        "rng_cuda": torch.cuda.get_rng_state_all() if DEVICE == "cuda" else None}, tmp)
            os.replace(tmp, ckpt_path)

    info["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
    json.dump(info, open(f"{run_dir}/run_info.json", "w"), indent=2, ensure_ascii=False)
    open(f"{run_dir}/DONE", "w").write(info["finished"])
    shutil.rmtree(os.path.dirname(ckpt_path), ignore_errors=True)
    del model, opt; torch.cuda.empty_cache() if DEVICE == "cuda" else None

## 7. Smoke test
1 epoch on 256 AJGT tweets per model: checks the full pipeline and prints time per epoch. Written to a separate `outputs/smoke` folder.

In [ ]:
RUN_SMOKE_TEST = True
if RUN_SMOKE_TEST:
    for mk in MODELS:
        shutil.rmtree(f"{PROJECT_DIR}/outputs/smoke/ajgt/{mk}", ignore_errors=True)
        train_one_run("ajgt", mk, seed=0, out_root=f"{PROJECT_DIR}/outputs/smoke",
                      ckpt_root=f"{PROJECT_DIR}/checkpoints/smoke", max_train=256, epochs=1)

## 8. All historical runs (2 models × 3 datasets × 3 seeds = 18)
Order is **seed-first**: all 6 dataset×model runs for seed 42, then 43, then 44, so a complete first-seed table exists early.
To split work across several Colab/Kaggle accounts, give each account a different `ONLY` list and copy the `outputs/historical` folders together at the end.
Safe to re-run at any time: finished runs are skipped, interrupted runs resume.
To run only part of the grid in one session, set `ONLY`, e.g. `ONLY = [("astd", "marbert", 42)]`.

In [ ]:
RUN_ALL = True
ONLY = None   # e.g. [("astd", "marbert", 42), ("astd", "arabert", 42)]

if RUN_ALL:
    grid = [(d, m, s) for s in SEEDS for d in RUN_ORDER for m in MODELS]   # seed-first: one full seed of results ASAP
    if ONLY: grid = [g for g in grid if g in ONLY]
    for d, m, s in grid:
        train_one_run(d, m, s)

Rerun Auther Agjt

In [8]:
# AJGT with the authors' own train/test split (UBC-NLP/marbert, examples/)
import subprocess, os, pandas as pd
src = "/content/marbert_repo"
if not os.path.exists(src):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/UBC-NLP/marbert", src], check=True)
os.makedirs(f"{DATA_DIR}/ajgt_ubc", exist_ok=True)
for sp in ["train", "test"]:
    d = pd.read_csv(f"{src}/examples/UBC_AJGT_final_shuffled_{sp}.tsv", sep="\t", dtype=str, quoting=3)
    d.rename(columns={"content": "text"})[["text", "label"]].to_csv(f"{DATA_DIR}/ajgt_ubc/{sp}.csv", index=False)
    print(sp, len(d), d["label"].value_counts().to_dict())   # expect train 1440, test 360
DATASETS["ajgt_ubc"] = dict(DATASETS["ajgt"])               # same labels, metric and published scores
for s in SEEDS:
    for mk in MODELS:
        train_one_run("ajgt_ubc", mk, s)

train 1440 {'Negative': 731, 'Positive': 709}
test 360 {'Positive': 191, 'Negative': 169}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider

ajgt_ubc/marbert/seed42 ep01 loss=0.6739 accuracy=85.83 macroF1=85.83 (5.7s)
ajgt_ubc/marbert/seed42 ep02 loss=0.5069 accuracy=91.11 macroF1=91.10 (5.3s)
ajgt_ubc/marbert/seed42 ep03 loss=0.2362 accuracy=93.33 macroF1=93.31 (5.4s)
ajgt_ubc/marbert/seed42 ep04 loss=0.1399 accuracy=95.28 macroF1=95.25 (5.3s)
ajgt_ubc/marbert/seed42 ep05 loss=0.1107 accuracy=94.44 macroF1=94.43 (5.3s)
ajgt_ubc/marbert/seed42 ep06 loss=0.0796 accuracy=94.72 macroF1=94.69 (5.3s)
ajgt_ubc/marbert/seed42 ep07 loss=0.0683 accuracy=95.00 macroF1=94.97 (5.6s)
ajgt_ubc/marbert/seed42 ep08 loss=0.0522 accuracy=95.00 macroF1=94.97 (5.5s)
ajgt_ubc/marbert/seed42 ep09 loss=0.0420 accuracy=95.28 macroF1=95.25 (5.3s)
ajgt_ubc/marbert/seed42 ep10 loss=0.0381 accuracy=94.44 macroF1=94.43 (5.3s)
ajgt_ubc/marbert/seed42 ep11 loss=0.0285 accuracy=94.72 macroF1=94.70 (5.5s)
ajgt_ubc/marbert/seed42 ep12 loss=0.0225 accuracy=95.00 macroF1=94.98 (5.5s)
ajgt_ubc/marbert/seed42 ep13 loss=0.0240 accuracy=95.00 macroF1=94.97 (5.5s)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv01
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (9 outputs left out)

ajgt_ubc/marbert/seed44 ep01 loss=0.6691 accuracy=85.00 macroF1=84.96 (5.3s)
ajgt_ubc/marbert/seed44 ep02 loss=0.5117 accuracy=91.94 macroF1=91.92 (5.5s)
ajgt_ubc/marbert/seed44 ep03 loss=0.2550 accuracy=94.72 macroF1=94.70 (5.4s)
ajgt_ubc/marbert/seed44 ep04 loss=0.1508 accuracy=94.72 macroF1=94.70 (5.3s)
ajgt_ubc/marbert/seed44 ep05 loss=0.1030 accuracy=94.72 macroF1=94.70 (5.4s)
ajgt_ubc/marbert/seed44 ep06 loss=0.0751 accuracy=95.56 macroF1=95.54 (5.4s)
ajgt_ubc/marbert/seed44 ep07 loss=0.0659 accuracy=95.28 macroF1=95.26 (5.4s)
ajgt_ubc/marbert/seed44 ep08 loss=0.0536 accuracy=94.72 macroF1=94.71 (5.3s)
ajgt_ubc/marbert/seed44 ep09 loss=0.0472 accuracy=95.28 macroF1=95.26 (5.4s)
ajgt_ubc/marbert/seed44 ep10 loss=0.0380 accuracy=95.00 macroF1=94.99 (5.5s)
ajgt_ubc/marbert/seed44 ep11 loss=0.0337 accuracy=95.28 macroF1=95.26 (5.5s)
ajgt_ubc/marbert/seed44 ep12 loss=0.0287 accuracy=95.28 macroF1=95.26 (5.6s)
ajgt_ubc/marbert/seed44 ep13 loss=0.0320 accuracy=94.72 macroF1=94.71 (5.4s)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv01
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ajgt_ubc/arabert/seed44 ep01 loss=0.6475 accuracy=82.22 macroF1=82.22 (5.1s)
ajgt_ubc/arabert/seed44 ep02 loss=0.4742 accuracy=85.56 macroF1=85.55 (5.1s)
ajgt_ubc/arabert/seed44 ep03 loss=0.3537 accuracy=88.61 macroF1=88.54 (5.1s)
ajgt_ubc/arabert/seed44 ep04 loss=0.2706 accuracy=89.17 macroF1=89.09 (5.2s)
ajgt_ubc/arabert/seed44 ep05 loss=0.2335 accuracy=90.00 macroF1=89.96 (5.1s)
ajgt_ubc/arabert/seed44 ep06 loss=0.1991 accuracy=90.56 macroF1=90.53 (5.0s)
ajgt_ubc/arabert/seed44 ep07 loss=0.1770 accuracy=90.28 macroF1=90.21 (5.2s)
ajgt_ubc/arabert/seed44 ep08 loss=0.1407 accuracy=91.94 macroF1=91.91 (5.2s)
ajgt_ubc/arabert/seed44 ep09 loss=0.1248 accuracy=91.94 macroF1=91.91 (5.1s)
ajgt_ubc/arabert/seed44 ep10 loss=0.1139 accuracy=92.50 macroF1=92.46 (5.1s)
ajgt_ubc/arabert/seed44 ep11 loss=0.0899 accuracy=92.22 macroF1=92.17 (5.1s)
ajgt_ubc/arabert/seed44 ep12 loss=0.0879 accuracy=92.50 macroF1=92.44 (5.1s)
ajgt_ubc/arabert/seed44 ep13 loss=0.0766 accuracy=91.39 macroF1=91.34 (5.1s)

## 9. Published vs ours: reproduction verdict
For each run the **best epoch on test** (historical metric; earliest epoch on ties) is taken.
A target is **reproduced** if |gap| ≤ 2 pp **or** the published score lies within our seed range,
**and** MARBERT > AraBERT (mean over seeds) on that dataset.

In [9]:
def collect_runs(out_root=OUT_DIR):
    rows = []
    for d in DATASETS:
        key = DATASETS[d]["metric"]
        for mk in MODELS:
            for s in SEEDS:
                rd = f"{out_root}/{d}/{mk}/seed{s}"
                if not os.path.exists(f"{rd}/DONE"): continue
                ep = [json.loads(l) for l in open(f"{rd}/metrics_per_epoch.jsonl")]
                best = max(ep, key=lambda e: (e[key], -e["epoch"]))
                rows.append(dict(dataset=d, model=mk, seed=s, metric=key, best_epoch=best["epoch"],
                                 score=best[key], macro_f1_at_best=best["macro_f1"], accuracy_at_best=best["accuracy"]))
    return pd.DataFrame(rows)

runs = collect_runs()
if runs.empty:
    print("No finished runs yet.")
else:
    runs.to_csv(f"{OUT_DIR}/per_run_best_epoch.csv", index=False)
    display(runs) if IN_COLAB else print(runs.to_string())

    summ = []
    for d in DATASETS:
        means = {}
        for mk in MODELS:
            r = runs[(runs.dataset == d) & (runs.model == mk)]
            if r.empty: continue
            pub = DATASETS[d]["published"][mk]
            mean, sd = r.score.mean(), (r.score.std(ddof=1) if len(r) > 1 else float("nan"))
            means[mk] = mean
            summ.append(dict(dataset=d, metric=DATASETS[d]["metric"], model=mk, n_seeds=len(r),
                             published=pub, ours_mean=round(mean, 2), ours_sd=round(sd, 2),
                             ours_min=round(r.score.min(), 2), ours_max=round(r.score.max(), 2),
                             gap_pp=round(mean - pub, 2),
                             within_2pp=abs(mean - pub) <= 2, within_seed_range=r.score.min() <= pub <= r.score.max()))
        ranking = means.get("marbert", np.nan) > means.get("arabert", np.nan) if len(means) == 2 else None
        for row in summ:
            if row["dataset"] == d:
                row["marbert_gt_arabert"] = ranking
                row["reproduced"] = bool((row["within_2pp"] or row["within_seed_range"]) and ranking) if ranking is not None else None
    summary = pd.DataFrame(summ)
    summary.to_csv(f"{OUT_DIR}/reproduction_summary.csv", index=False)
    display(summary) if IN_COLAB else print(summary.to_string())
    incomplete = summary[summary.n_seeds < len(SEEDS)]
    if not incomplete.empty: print("NOTE: some targets have fewer than", len(SEEDS), "seeds - verdict is provisional.")

,dataset,model,seed,metric,best_epoch,score,macro_f1_at_best,accuracy_at_best
0,arsarcasm,marbert,42,f1_pn,5,72.113760,74.748703,77.109005
1,arsarcasm,marbert,43,f1_pn,11,73.176557,75.457292,77.535545
2,arsarcasm,marbert,44,f1_pn,10,71.973836,74.236525,76.303318
3,arsarcasm,arabert,42,f1_pn,12,67.630053,70.793725,73.554502
4,arsarcasm,arabert,43,f1_pn,10,68.216314,71.906440,75.260664
5,arsarcasm,arabert,44,f1_pn,13,67.740980,71.173040,74.360190
6,astd,marbert,42,f1_pn,10,81.515484,69.237273,73.071104
7,astd,marbert,43,f1_pn,6,80.483165,68.179253,72.465961
8,astd,marbert,44,f1_pn,5,80.688823,67.326384,72.314675
9,astd,arabert,42,f1_pn,17,75.797348,64.094784,68.229955


,dataset,metric,model,n_seeds,published,ours_mean,ours_sd,ours_min,ours_max,gap_pp,within_2pp,within_seed_range,marbert_gt_arabert,reproduced
0,arsarcasm,f1_pn,marbert,3,71.50,72.42,0.66,71.97,73.18,0.92,True,False,True,True
1,arsarcasm,f1_pn,arabert,3,63.50,67.86,0.31,67.63,68.22,4.36,False,False,True,False
2,astd,f1_pn,marbert,3,78.00,80.90,0.55,80.48,81.52,2.90,False,False,True,False
3,astd,f1_pn,arabert,3,72.00,75.83,0.17,75.68,76.01,3.83,False,False,True,False
4,ajgt,accuracy,marbert,3,96.11,94.35,0.32,94.17,94.72,-1.76,True,False,True,True
5,ajgt,accuracy,arabert,3,92.22,88.89,0.73,88.06,89.44,-3.33,False,False,True,False
6,ajgt_ubc,accuracy,marbert,3,96.11,95.83,0.28,95.56,96.11,-0.28,True,True,True,True
7,ajgt_ubc,accuracy,arabert,3,92.22,92.13,0.32,91.94,92.50,-0.09,True,True,True,True


## 10. Departures log
Creates `departures.md` in Drive the first time (edit it there; it is not overwritten).

In [ ]:
DEP_FILE = f"{PROJECT_DIR}/departures.md"
if not os.path.exists(DEP_FILE):
    open(DEP_FILE, "w").write(f"""# Departures and assumptions - historical reproduction

| # | Paper [1] | Ours | Why / expected effect |
|---|---|---|---|
| 1 | AraBERT version not named | `{MODELS['arabert']}` @ `{REVISIONS['arabert']}` | Fixed 2020 base release; possible source of AraBERT gaps |
| 2 | MARBERT | `{MODELS['marbert']}` @ `{REVISIONS['marbert']}` | Original release, not MARBERTv2 |
| 3 | LR scheduler / warm-up unreported | Constant LR {HP['lr']}, no warm-up | Simplest reading of the paper |
| 4 | Precision not reported | fp16 mixed precision = {USE_AMP} | Speed on T4; small numeric differences |
| 5 | Padding not reported | Dynamic padding (max_len {HP['max_len']} truncation kept) | Speed; no effect on outputs with attention masks beyond float noise |
| 6 | Fine-tuning preprocessing not reported | mentions -> USER, URLs -> URL: {PREPROC} | Mirrors MARBERT pre-training normalisation (verify) |
| 7 | Seeds not reported | {SEEDS} | Three seeds per configuration |
| 8 | AJGT split not released | (fill in with Member 1) | Most likely cause of AJGT gaps |
| 9 | Hardware not reported | {gpu_info} | GPU non-determinism; small run-to-run noise |
| 10 | Classification head | HF `AutoModelForSequenceClassification` default (dropout 0.1, [CLS] pooler) | Standard BERT fine-tuning |
""")
    print("Created", DEP_FILE)
else:
    print(DEP_FILE, "exists - edit it in Drive")

---
# Independent protocol: dev-selected runs (audit item 3, Member 3)
Everything above is the **historical** protocol and is left unchanged. The sections below add the
**independent** protocol: train on 90% of the training set, **choose the epoch on the held-out dev set**,
and report the **test** score at that epoch. Models, hyperparameters, seeds and test sets are identical,
so the only difference between the two protocols is how the epoch is chosen.

**Before running:** run sections 1–6 first (setup, config, data, metrics, seeding, the training helpers;
set `RUN_SMOKE_TEST = False` in section 7). Section 8 can be skipped: finished historical runs are skipped anyway.

## 11. Dev-split data (Member 1's files)
Upload Member 1's nine files **as they are** into one Drive folder:
```
marbert_repro/data_dev/
  arsarcasm_train.csv  arsarcasm_dev.csv  arsarcasm_test.csv
  astd_train.csv       astd_dev.csv       astd_test.csv
  ajgt_train.csv       ajgt_dev.csv       ajgt_test.csv
```
(`*_train_full.csv` is not needed: it equals the historical training data.) Only the `text` and `label`
columns are used; `id` and the other columns are kept for error analysis.

In [10]:
DEV_DATA_DIR = f"{PROJECT_DIR}/data_dev"
OUT_DIR_DEV  = f"{PROJECT_DIR}/outputs/independent"
CKPT_DIR_DEV = f"{PROJECT_DIR}/checkpoints/independent"

def load_split_dev(dataset, split):
    """Same cleaning and label checks as load_split (section 3), reading Member 1's files."""
    path = f"{DEV_DATA_DIR}/{dataset}_{split}.csv"
    df = pd.read_csv(path)
    missing = {TEXT_COL, LABEL_COL} - set(df.columns)
    assert not missing, f"{path} is missing columns {missing}"
    keep = df.drop(columns=[c for c in ["text_clean"] if c in df.columns])
    raw = keep[LABEL_COL].astype(str).str.strip().str.lower()
    labels = DATASETS[dataset]["labels"]
    out = pd.DataFrame({"id": keep["id"] if "id" in keep.columns else np.arange(len(keep)),
                        "label": raw.map(LABEL_ALIASES)})
    bad = sorted(raw[out["label"].isna()].unique())
    assert not bad, f"{path}: unknown labels {bad}"
    extra = set(out["label"]) - set(labels)
    assert not extra, f"{path}: labels {extra} not expected for {dataset}"
    out["y"] = out["label"].map({l: i for i, l in enumerate(labels)})
    out["text"] = keep[TEXT_COL].map(preprocess)
    return out.reset_index(drop=True)

# Sanity check: sizes, label mix, and that train/dev/test do not share ids
for ds in DATASETS:
    parts = {}
    for sp in ["train", "dev", "test"]:
        p = f"{DEV_DATA_DIR}/{ds}_{sp}.csv"
        if not os.path.exists(p):
            print(f"{ds:10s} {sp:5s} MISSING -> {p}"); continue
        parts[sp] = load_split_dev(ds, sp)
        print(f"{ds:10s} {sp:5s} n={len(parts[sp]):6d}", parts[sp]["label"].value_counts().to_dict())
    if len(parts) == 3:
        assert not set(parts["train"]["id"]) & set(parts["dev"]["id"]), f"{ds}: train and dev share ids"
        hist_test = f"{DATA_DIR}/{ds}/test.csv"
        if os.path.exists(hist_test):
            same = sorted(pd.read_csv(hist_test)["text"].astype(str)) == sorted(pd.read_csv(f"{DEV_DATA_DIR}/{ds}_test.csv")[TEXT_COL].astype(str))
            print(f"{'':10s} test identical to historical test: {same}")

arsarcasm  train n=  7593 {'neutral': 3836, 'negative': 2531, 'positive': 1226}
arsarcasm  dev   n=   844 {'neutral': 426, 'negative': 282, 'positive': 136}
arsarcasm  test  n=  2110 {'neutral': 1078, 'negative': 716, 'positive': 316}
           test identical to historical test: True
astd       train n= 24654 {'negative': 10118, 'neutral': 8534, 'positive': 6002}
astd       dev   n=  2740 {'negative': 1123, 'neutral': 949, 'positive': 668}
astd       test  n=   661 {'negative': 336, 'neutral': 166, 'positive': 159}
           test identical to historical test: True
ajgt       train n=  1296 {'negative': 648, 'positive': 648}
ajgt       dev   n=   144 {'positive': 72, 'negative': 72}
ajgt       test  n=   360 {'positive': 180, 'negative': 180}
           test identical to historical test: True
ajgt_ubc   train MISSING -> /content/drive/MyDrive/marbert_repro/data_dev/ajgt_ubc_train.csv
ajgt_ubc   dev   MISSING -> /content/drive/MyDrive/marbert_repro/data_dev/ajgt_ubc_dev.csv
ajgt_ubc   

## 12. One dev-selected run (resumable)
Same training loop and settings as section 6. After every epoch it scores **dev and test**, and writes
`metrics_per_epoch.jsonl` (each line holds `dev` and `test` metrics) plus dev and test predictions.
The epoch is chosen afterwards (section 14) on **dev only**.

In [ ]:
def train_one_run_dev(dataset, model_key, seed, out_root=None, ckpt_root=None, max_train=None, epochs=None):
    out_root, ckpt_root = out_root or OUT_DIR_DEV, ckpt_root or CKPT_DIR_DEV
    epochs = epochs or HP["epochs"]
    labels = DATASETS[dataset]["labels"]
    run_dir   = f"{out_root}/{dataset}/{model_key}/seed{seed}"
    ckpt_path = f"{ckpt_root}/{dataset}/{model_key}/seed{seed}/last.pt"
    if os.path.exists(f"{run_dir}/DONE"):
        print(f"[skip] {dataset}/{model_key}/seed{seed} already done"); return
    os.makedirs(f"{run_dir}/preds", exist_ok=True); os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

    set_seed(seed)
    train_df, dev_df, test_df = (load_split_dev(dataset, s) for s in ["train", "dev", "test"])
    if max_train: train_df = train_df.sample(n=min(max_train, len(train_df)), random_state=seed).reset_index(drop=True)

    mid, rev = MODELS[model_key], REVISIONS[model_key]
    tok = AutoTokenizer.from_pretrained(mid, revision=rev)
    model = AutoModelForSequenceClassification.from_pretrained(mid, revision=rev, num_labels=len(labels)).to(DEVICE)
    enc = lambda df: tok(df["text"].tolist(), truncation=True, max_length=HP["max_len"])
    train_ds = EncDataset(enc(train_df), train_df["y"].values)
    collate = DataCollatorWithPadding(tok)
    dev_loader  = DataLoader(EncDataset(enc(dev_df),  dev_df["y"].values),  batch_size=128, shuffle=False, collate_fn=collate)
    test_loader = DataLoader(EncDataset(enc(test_df), test_df["y"].values), batch_size=128, shuffle=False, collate_fn=collate)

    opt = torch.optim.Adam(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    start_epoch, metrics_file = 1, f"{run_dir}/metrics_per_epoch.jsonl"

    if os.path.exists(ckpt_path):                                    # ---- resume
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        torch.set_rng_state(ck["rng_cpu"])
        if DEVICE == "cuda" and ck.get("rng_cuda") is not None: torch.cuda.set_rng_state_all(ck["rng_cuda"])
        start_epoch = ck["epoch"] + 1
        if os.path.exists(metrics_file):
            keep = [l for l in open(metrics_file) if json.loads(l)["epoch"] <= ck["epoch"]]
            open(metrics_file, "w").writelines(keep)
        print(f"[resume] {dataset}/{model_key}/seed{seed} from epoch {start_epoch}")
    elif os.path.exists(metrics_file):
        os.remove(metrics_file)

    info = dict(protocol="independent: train = 90% of training set, best epoch selected on DEV, test reported at that epoch",
                dataset=dataset, model_key=model_key, model_id=mid, model_revision=rev, seed=seed, labels=labels,
                n_train=len(train_df), n_dev=len(dev_df), n_test=len(test_df),
                md5={s: file_md5(f"{DEV_DATA_DIR}/{dataset}_{s}.csv") for s in ["train", "dev", "test"]},
                hp=HP, epochs=epochs, use_amp=USE_AMP, preprocessing=PREPROC, dynamic_padding=True, max_train=max_train,
                gpu=gpu_info, torch=torch.__version__, transformers=transformers.__version__,
                started=time.strftime("%Y-%m-%d %H:%M:%S"))
    json.dump(info, open(f"{run_dir}/run_info.json", "w"), indent=2, ensure_ascii=False)
    key = DATASETS[dataset]["metric"]

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time(); model.train()
        g = torch.Generator(); g.manual_seed(seed * 1000 + epoch)
        loader = DataLoader(train_ds, batch_size=HP["batch_size"], shuffle=True, generator=g, collate_fn=collate)
        tot, n = 0.0, 0
        for b in loader:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            opt.zero_grad(set_to_none=True)
            with _autocast():
                loss = model(**b).loss
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * b["labels"].size(0); n += b["labels"].size(0)

        rec = {"epoch": epoch, "train_loss": tot / n}
        for split, loader_, df in [("dev", dev_loader, dev_df), ("test", test_loader, test_df)]:
            probs = predict(model, loader_); pred = probs.argmax(1)
            rec[split] = compute_metrics(df["y"].values, pred, labels)
            pdf = pd.DataFrame({"id": df["id"].values, "gold": df["y"].values, "pred": pred})
            for i, l in enumerate(labels): pdf[f"p_{l}"] = probs[:, i].round(6)
            pdf.to_csv(f"{run_dir}/preds/epoch_{epoch:02d}_{split}_preds.csv", index=False)
        rec["seconds"] = round(time.time() - t0, 1)
        with open(metrics_file, "a") as f: f.write(json.dumps(rec) + "\n")
        print(f"[dev-sel] {dataset}/{model_key}/seed{seed} ep{epoch:02d} loss={tot/n:.4f} "
              f"dev {key}={rec['dev'][key]:.2f}  test {key}={rec['test'][key]:.2f} ({rec['seconds']}s)")

        ce = CKPT_EVERY.get(dataset, 1) if isinstance(CKPT_EVERY, dict) else CKPT_EVERY
        if ce and epoch % ce == 0 and epoch < epochs:
            tmp = ckpt_path + ".tmp"
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "epoch": epoch, "rng_cpu": torch.get_rng_state(),
                        "rng_cuda": torch.cuda.get_rng_state_all() if DEVICE == "cuda" else None}, tmp)
            os.replace(tmp, ckpt_path)

    info["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
    json.dump(info, open(f"{run_dir}/run_info.json", "w"), indent=2, ensure_ascii=False)
    open(f"{run_dir}/DONE", "w").write(info["finished"])
    shutil.rmtree(os.path.dirname(ckpt_path), ignore_errors=True)
    del model, opt; torch.cuda.empty_cache() if DEVICE == "cuda" else None

## 13. All dev-selected runs (2 models × 3 datasets × 3 seeds = 18)
Seed-first order, resumable, and `ONLY_DEV` works like `ONLY` in section 8.
Optional smoke test first: 1 epoch on 256 AJGT tweets per model, written to `outputs/smoke_independent`.

In [ ]:
RUN_DEV_SMOKE_TEST = True
if RUN_DEV_SMOKE_TEST:
    for mk in MODELS:
        shutil.rmtree(f"{PROJECT_DIR}/outputs/smoke_independent/ajgt/{mk}", ignore_errors=True)
        train_one_run_dev("ajgt", mk, seed=0, out_root=f"{PROJECT_DIR}/outputs/smoke_independent",
                          ckpt_root=f"{PROJECT_DIR}/checkpoints/smoke_independent", max_train=256, epochs=1)

RUN_ALL_DEV = True
ONLY_DEV = None   # e.g. [("astd", "marbert", 42)]
if RUN_ALL_DEV:
    grid = [(d, m, s) for s in SEEDS for d in RUN_ORDER for m in MODELS]
    if ONLY_DEV: grid = [g for g in grid if g in ONLY_DEV]
    for d, m, s in grid:
        train_one_run_dev(d, m, s)

## 14. Dev-selected results and comparison with the historical protocol
For each run: the epoch with the best **dev** score (historical metric; earliest on ties), and the **test**
score at that epoch. Then, per target: mean ± SD over seeds, the gap to [1], the MARBERT > AraBERT check,
and **inflation** = historical best-on-test mean − dev-selected mean (how much choosing on test raised the score).

In [ ]:
def collect_dev_runs(out_root=OUT_DIR_DEV):
    rows = []
    for d in DATASETS:
        key = DATASETS[d]["metric"]
        for mk in MODELS:
            for s in SEEDS:
                rd = f"{out_root}/{d}/{mk}/seed{s}"
                if not os.path.exists(f"{rd}/DONE"): continue
                ep = [json.loads(l) for l in open(f"{rd}/metrics_per_epoch.jsonl")]
                sel = max(ep, key=lambda e: (e["dev"][key], -e["epoch"]))           # chosen on DEV
                oracle = max(ep, key=lambda e: (e["test"][key], -e["epoch"]))       # best-on-test, same runs (reference)
                rows.append(dict(dataset=d, model=mk, seed=s, metric=key,
                                 dev_epoch=sel["epoch"], dev_score=sel["dev"][key], test_at_dev_epoch=sel["test"][key],
                                 macro_f1_at_dev_epoch=sel["test"]["macro_f1"],
                                 best_test_epoch=oracle["epoch"], best_test_score=oracle["test"][key]))
    return pd.DataFrame(rows)

dev_runs = collect_dev_runs()
if dev_runs.empty:
    print("No finished dev-selected runs yet.")
else:
    dev_runs.to_csv(f"{OUT_DIR_DEV}/per_run_dev_selected.csv", index=False)
    display(dev_runs.round(2)) if IN_COLAB else print(dev_runs.round(2).to_string())

    hist = collect_runs() if os.path.exists(OUT_DIR) else pd.DataFrame()
    summ = []
    for d in DATASETS:
        means = {}
        for mk in MODELS:
            r = dev_runs[(dev_runs.dataset == d) & (dev_runs.model == mk)]
            if r.empty: continue
            pub = DATASETS[d]["published"][mk]
            mean = r.test_at_dev_epoch.mean(); sd = r.test_at_dev_epoch.std(ddof=1) if len(r) > 1 else float("nan")
            means[mk] = mean
            h = hist[(hist.dataset == d) & (hist.model == mk)] if not hist.empty else pd.DataFrame()
            h_mean = h.score.mean() if not h.empty else float("nan")
            summ.append(dict(dataset=d, metric=DATASETS[d]["metric"], model=mk, n_seeds=len(r), published=pub,
                             dev_selected_mean=round(mean, 2), dev_selected_sd=round(sd, 2),
                             gap_to_paper=round(mean - pub, 2),
                             historical_mean=round(h_mean, 2), inflation=round(h_mean - mean, 2),
                             same_runs_best_test_mean=round(r.best_test_score.mean(), 2)))
        rank = means.get("marbert", np.nan) > means.get("arabert", np.nan) if len(means) == 2 else None
        for row in summ:
            if row["dataset"] == d: row["marbert_gt_arabert"] = rank
    dev_summary = pd.DataFrame(summ)
    dev_summary.to_csv(f"{OUT_DIR_DEV}/independent_summary.csv", index=False)
    display(dev_summary) if IN_COLAB else print(dev_summary.to_string())
    if (dev_summary.n_seeds < len(SEEDS)).any(): print("NOTE: some targets have fewer than", len(SEEDS), "seeds - provisional.")

## 15. Export predictions for error analysis (Member 1)
One CSV with a row per test tweet, per run, at the **selected epoch** of each protocol
(historical: best epoch on test; independent: best epoch on dev). Columns:
`dataset, model, protocol, seed, epoch, id, gold, pred, correct` (+ class probabilities).
Historical prediction files store the test row number; it maps to Member 1's `id` because the test files
have identical row order (checked below). Written to `outputs/error_analysis_predictions.csv`.

In [8]:
def export_error_analysis(path=f"{PROJECT_DIR}/outputs/error_analysis_predictions.csv"):
    frames = []
    for ds in DATASETS:
        labels, key = DATASETS[ds]["labels"], DATASETS[ds]["metric"]
        m1_test = f"{DEV_DATA_DIR}/{ds}_test.csv"
        ids = pd.read_csv(m1_test)["id"] if os.path.exists(m1_test) else None
        if ids is not None and os.path.exists(f"{DATA_DIR}/{ds}/test.csv"):   # row order must match to map idx -> id
            same = (pd.read_csv(f"{DATA_DIR}/{ds}/test.csv")["text"].astype(str).values
                    == pd.read_csv(m1_test)[TEXT_COL].astype(str).values).all()
            assert same, f"{ds}: historical and Member 1 test files are not in the same row order"
        for protocol, root in [("historical", OUT_DIR), ("independent", OUT_DIR_DEV)]:
            for mk in MODELS:
                for s in SEEDS:
                    rd = f"{root}/{ds}/{mk}/seed{s}"
                    if not os.path.exists(f"{rd}/DONE"): continue
                    ep = [json.loads(l) for l in open(f"{rd}/metrics_per_epoch.jsonl")]
                    if protocol == "historical":
                        best = max(ep, key=lambda e: (e[key], -e["epoch"]))["epoch"]
                        p = pd.read_csv(f"{rd}/preds/epoch_{best:02d}_test_preds.csv")
                        p["id"] = ids.values[p["idx"].values] if ids is not None else p["idx"]
                    else:
                        best = max(ep, key=lambda e: (e["dev"][key], -e["epoch"]))["epoch"]
                        p = pd.read_csv(f"{rd}/preds/epoch_{best:02d}_test_preds.csv")
                    out = pd.DataFrame({"dataset": ds, "model": mk, "protocol": protocol, "seed": s, "epoch": best,
                                        "id": p["id"].values,
                                        "gold": [labels[g] for g in p["gold"]], "pred": [labels[q] for q in p["pred"]]})
                    out["correct"] = out["gold"] == out["pred"]
                    for l in labels: out[f"p_{l}"] = p[f"p_{l}"].values
                    frames.append(out)
    if not frames:
        print("No finished runs to export."); return None
    df = pd.concat(frames, ignore_index=True)
    df.to_csv(path, index=False)
    print(f"Wrote {len(df):,} rows -> {path}")
    print(df.groupby(["protocol", "dataset", "model"]).agg(runs=("seed", "nunique"), accuracy=("correct", "mean")).round(4))
    return df

ea = export_error_analysis()

NameError: name 'DEV_DATA_DIR' is not defined

---
# Extra analyses (optional): sensitivity tests, AraBERT versions, linguistic challenge set
Everything above is unchanged. These sections add **clearly labelled extra experiments**; they do not replace
the main results. All extras use **seed 42 on ArSarcasm-Sent** to stay cheap (about 15–20 min per run on an L4).

| Tag | What changes | Why | Protocol | Compared with |
|---|---|---|---|---|
| `sched_linear_warmup` | LR schedule: linear decay + 10% warm-up (both models) | Schedule unreported in [1]; ours is constant | historical | historical seed-42 runs |
| `lr_2e-5` | Learning rate 2e-5 instead of 2e-6 (both models) | Does the ranking survive a standard BERT rate? | historical | historical seed-42 runs |
| `arabert_v01_official_prep` | AraBERT v0.1 with AraBERT's own preprocessing | Tests the preprocessing explanation for the AraBERT gap | historical | historical seed-42 AraBERT |
| `arabert_v1_farasa` | AraBERT v1 (`bert-base-arabert`) + Farasa segmentation | Tests the version explanation for the AraBERT gap | historical | historical seed-42 AraBERT |
| `challenge` | Also scores the validated challenge set every epoch (both models) | Linguistic challenge set (section 18) | independent (dev-selected) | — |

Run sections 1–6 and 11 first (set the training flags in sections 7, 8 and 13 to `False`).

## 16. Extra-experiment runner

In [19]:
import math
from transformers import get_linear_schedule_with_warmup
OUT_DIR_EXTRA = f"{PROJECT_DIR}/outputs/extras"
CHALLENGE_DIR = f"{PROJECT_DIR}/challenge"

_TEXT_FNS = {}
def get_text_fn(spec):
    """None -> our preprocessing (section 3). 'arabert:<model name>' -> AraBERT's official ArabertPreprocessor."""
    if spec is None: return preprocess
    if spec in _TEXT_FNS: return _TEXT_FNS[spec]
    if spec.startswith("arabert:"):
        name = spec.split(":", 1)[1]
        try:
            from arabert.preprocess import ArabertPreprocessor
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arabert"], check=True)
            from arabert.preprocess import ArabertPreprocessor
        if "arabertv0" not in name and shutil.which("java") is None:      # v1/v2 use Farasa segmentation (Java)
            subprocess.run(["apt-get", "install", "-y", "-q", "default-jre-headless"], check=True)
        prep = ArabertPreprocessor(model_name=name)
        _TEXT_FNS[spec] = lambda t: prep.preprocess(str(t))
        return _TEXT_FNS[spec]
    raise ValueError(spec)

def _read(path, dataset, text_fn):
    df = pd.read_csv(path)
    raw = df[LABEL_COL].astype(str).str.strip().str.lower().map(LABEL_ALIASES)
    assert raw.notna().all(), f"{path}: unknown labels"
    out = pd.DataFrame({"id": df["id"] if "id" in df.columns else np.arange(len(df)), "label": raw})
    out["y"] = out["label"].map({l: i for i, l in enumerate(DATASETS[dataset]["labels"])})
    out["text"] = [text_fn(t) for t in df[TEXT_COL]]
    return out

def train_one_run_extra(tag, dataset, model_key, seed, model_id=None, lr=None, schedule="constant", warmup=0.0,
                        protocol="historical", text_fn=None, challenge=False, epochs=None, max_train=None, out_root=None):
    out_root = out_root or OUT_DIR_EXTRA
    epochs, lr = epochs or HP["epochs"], lr or HP["lr"]
    labels, key = DATASETS[dataset]["labels"], DATASETS[dataset]["metric"]
    run_dir = f"{out_root}/{tag}/{dataset}/{model_key}/seed{seed}"
    if os.path.exists(f"{run_dir}/DONE"):
        print(f"[skip] extra {tag}/{dataset}/{model_key}/seed{seed} done"); return
    os.makedirs(f"{run_dir}/preds", exist_ok=True)
    fn = get_text_fn(text_fn)
    if protocol == "historical":
        train_df, dev_df = _read(f"{DATA_DIR}/{dataset}/train.csv", dataset, fn), None
        test_df = _read(f"{DATA_DIR}/{dataset}/test.csv", dataset, fn)
    else:
        train_df, dev_df, test_df = (_read(f"{DEV_DATA_DIR}/{dataset}_{s}.csv", dataset, fn) for s in ["train", "dev", "test"])
    ch = None
    if challenge:
        ch = pd.read_csv(f"{CHALLENGE_DIR}/challenge_validated.csv")
        ch["y"] = ch["gold"].map({l: i for i, l in enumerate(labels)})
    set_seed(seed)
    if max_train: train_df = train_df.sample(n=min(max_train, len(train_df)), random_state=seed).reset_index(drop=True)

    mid = model_id or MODELS[model_key]
    rev = REVISIONS[model_key] if model_id is None else (None if os.path.isdir(mid) else __import__("huggingface_hub").HfApi().model_info(mid).sha)
    tok = AutoTokenizer.from_pretrained(mid, revision=rev)
    model = AutoModelForSequenceClassification.from_pretrained(mid, revision=rev, num_labels=len(labels)).to(DEVICE)
    enc = lambda texts: tok(list(texts), truncation=True, max_length=HP["max_len"])
    collate = DataCollatorWithPadding(tok)
    mk_loader = lambda texts, y: DataLoader(EncDataset(enc(texts), y), batch_size=128, shuffle=False, collate_fn=collate)
    train_ds = EncDataset(enc(train_df["text"]), train_df["y"].values)
    evals = {"test": (mk_loader(test_df["text"], test_df["y"].values), test_df)}
    if dev_df is not None: evals["dev"] = (mk_loader(dev_df["text"], dev_df["y"].values), dev_df)
    if ch is not None:
        evals["ch_orig"] = (mk_loader([fn(t) for t in ch["source_text"]], ch["y"].values), ch)
        evals["ch_var"]  = (mk_loader([fn(t) for t in ch["variant_text"]], ch["y"].values), ch)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=HP["weight_decay"])
    steps = epochs * math.ceil(len(train_ds) / HP["batch_size"])
    sched = get_linear_schedule_with_warmup(opt, int(warmup * steps), steps) if schedule == "linear_warmup" else None
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    info = dict(tag=tag, protocol=protocol, dataset=dataset, model_key=model_key, model_id=mid, model_revision=rev, seed=seed,
                lr=lr, schedule=schedule, warmup=warmup, text_fn=text_fn or "ours (section 3)", challenge=bool(challenge),
                n_train=len(train_df), epochs=epochs, gpu=gpu_info, started=time.strftime("%Y-%m-%d %H:%M:%S"))
    json.dump(info, open(f"{run_dir}/run_info.json", "w"), indent=2, ensure_ascii=False)
    metrics_file = f"{run_dir}/metrics_per_epoch.jsonl"
    if os.path.exists(metrics_file): os.remove(metrics_file)          # extras restart from scratch if interrupted

    for epoch in range(1, epochs + 1):
        t0 = time.time(); model.train()
        g = torch.Generator(); g.manual_seed(seed * 1000 + epoch)
        tot, n = 0.0, 0
        for b in DataLoader(train_ds, batch_size=HP["batch_size"], shuffle=True, generator=g, collate_fn=collate):
            b = {k: v.to(DEVICE) for k, v in b.items()}
            opt.zero_grad(set_to_none=True)
            with _autocast():
                loss = model(**b).loss
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if sched is not None: sched.step()
            tot += loss.item() * b["labels"].size(0); n += b["labels"].size(0)
        rec = {"epoch": epoch, "train_loss": tot / n}
        for name, (loader, df) in evals.items():
            probs = predict(model, loader); pred = probs.argmax(1)
            if name.startswith("ch_"):
                pd.DataFrame({"pair_id": df["pair_id"], "gold": df["y"], "pred": pred}).to_csv(f"{run_dir}/preds/epoch_{epoch:02d}_{name}_preds.csv", index=False)
            else:
                rec[name] = compute_metrics(df["y"].values, pred, labels)
                pdf = pd.DataFrame({"id": df["id"].values, "gold": df["y"].values, "pred": pred})
                for i, l in enumerate(labels): pdf[f"p_{l}"] = probs[:, i].round(6)
                pdf.to_csv(f"{run_dir}/preds/epoch_{epoch:02d}_{name}_preds.csv", index=False)
        rec["seconds"] = round(time.time() - t0, 1)
        with open(metrics_file, "a") as f: f.write(json.dumps(rec) + "\n")
        dv = f" dev {key}={rec['dev'][key]:.2f}" if "dev" in rec else ""
        print(f"[extra:{tag}] {dataset}/{model_key}/seed{seed} ep{epoch:02d} loss={tot/n:.4f}{dv} test {key}={rec['test'][key]:.2f} ({rec['seconds']}s)")
    open(f"{run_dir}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
    del model, opt; torch.cuda.empty_cache() if DEVICE == "cuda" else None

## 17. Run the sensitivity and AraBERT-version experiments
Each entry runs for seed 42 on ArSarcasm-Sent. Turn individual experiments off by removing them from `EXTRAS`.
The `challenge` runs are in section 18, after the challenge set has been validated.

In [20]:
EXTRA_SEEDS = [42]
EXTRAS = [
    dict(tag="sched_linear_warmup", dataset="arsarcasm", models=["marbert", "arabert"], schedule="linear_warmup", warmup=0.1),
    dict(tag="lr_2e-5", dataset="arsarcasm", models=["marbert", "arabert"], lr=2e-5),
    #dict(tag="arabert_v01_official_prep", dataset="arsarcasm", models=["arabert"], text_fn="arabert:bert-base-arabertv01"),
    #dict(tag="arabert_v1_farasa", dataset="arsarcasm", models=["arabert"], model_id="aubmindlab/bert-base-arabert", text_fn="arabert:bert-base-arabert"),
]
RUN_EXTRAS = True
if RUN_EXTRAS:
    for e in EXTRAS:
        for s in EXTRA_SEEDS:
            for mk in e["models"]:
                kw = {k: v for k, v in e.items() if k not in ("tag", "dataset", "models")}
                try:
                    train_one_run_extra(e["tag"], e["dataset"], mk, s, **kw)
                except Exception as ex:                      # one failing extra (e.g. Farasa install) must not stop the others
                    print(f"[extra FAILED] {e['tag']}/{mk}: {type(ex).__name__}: {ex}")

def summarize_extras():
    rows = []
    for e in EXTRAS:
        d, key = e["dataset"], DATASETS[e["dataset"]]["metric"]
        for mk in e["models"]:
            for s in EXTRA_SEEDS:
                rd = f"{OUT_DIR_EXTRA}/{e['tag']}/{d}/{mk}/seed{s}"
                if not os.path.exists(f"{rd}/DONE"): continue
                ep = [json.loads(l) for l in open(f"{rd}/metrics_per_epoch.jsonl")]
                best = max(ep, key=lambda r: (r["test"][key], -r["epoch"]))           # historical protocol: best on test
                base_f = f"{OUT_DIR}/{d}/{mk}/seed{s}/metrics_per_epoch.jsonl"
                base = max((json.loads(l)[key] for l in open(base_f)), default=float("nan")) if os.path.exists(base_f) else float("nan")
                rows.append(dict(experiment=e["tag"], dataset=d, model=mk, seed=s, metric=key, best_epoch=best["epoch"],
                                 score=round(best["test"][key], 2), baseline_same_seed=round(base, 2),
                                 change=round(best["test"][key] - base, 2), published=DATASETS[d]["published"][mk]))
    df = pd.DataFrame(rows)
    if not df.empty:
        df.to_csv(f"{OUT_DIR_EXTRA}/extras_summary.csv", index=False)
        display(df) if IN_COLAB else print(df.to_string())
    else:
        print("No finished extra runs yet.")
    return df

extras_summary = summarize_extras()

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  654MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (11 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv01
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[extra:lr_2e-5] arsarcasm/arabert/seed42 ep01 loss=0.6937 test f1_pn=66.29 (27.2s)
[extra:lr_2e-5] arsarcasm/arabert/seed42 ep02 loss=0.4845 test f1_pn=66.63 (27.3s)
[extra:lr_2e-5] arsarcasm/arabert/seed42 ep03 loss=0.3266 test f1_pn=67.60 (27.3s)
[extra:lr_2e-5] arsarcasm/arabert/seed42 ep04 loss=0.2057 test f1_pn=65.95 (27.1s)
[extra:lr_2e-5] arsarcasm/arabert/seed42 ep05 loss=0.1327 test f1_pn=67.31 (27.1s)
[extra:lr_2e-5] arsarcasm/arabert/seed42 ep06 loss=0.0933 test f1_pn=65.48 (27.0s)
[extra:lr_2e-5] arsarcasm/arabert/seed42 ep07 loss=0.0703 test f1_pn=63.25 (26.8s)
[extra:lr_2e-5] arsarcasm/arabert/seed42 ep08 loss=0.0628 test f1_pn=64.74 (27.1s)
[extra:lr_2e-5] arsarcasm/arabert/seed42 ep09 loss=0.0514 test f1_pn=67.29 (27.2s)
[extra:lr_2e-5] arsarcasm/arabert/seed42 ep10 loss=0.0425 test f1_pn=65.07 (27.0s)
[extra:lr_2e-5] arsarcasm/arabert/seed42 ep11 loss=0.0461 test f1_pn=65.56 (27.0s)
[extra:lr_2e-5] arsarcasm/arabert/seed42 ep12 loss=0.0362 test f1_pn=67.48 (27.1s)
[ext

,experiment,dataset,model,seed,metric,best_epoch,score,baseline_same_seed,change,published
0,sched_linear_warmup,arsarcasm,marbert,42,f1_pn,9,72.17,72.11,0.05,71.5
1,sched_linear_warmup,arsarcasm,arabert,42,f1_pn,10,68.06,67.63,0.43,63.5
2,lr_2e-5,arsarcasm,marbert,42,f1_pn,3,71.67,72.11,-0.45,71.5
3,lr_2e-5,arsarcasm,arabert,42,f1_pn,3,67.60,67.63,-0.03,63.5


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Setup farasa


In [15]:
# --- AraBERT v1 + Farasa, using QCRI's Farasa release on GitHub (the farasapy download server is down) ---
# Run this cell after section 16, then re-run the section 17 cell. Replaces the earlier fix cell.
import os, re, shutil, subprocess, sys, tempfile

FARASA_ROOT = "/content/FarasaSegmenter" if IN_COLAB else os.path.abspath("FarasaSegmenter")
AR_WORD = re.compile(r"^[\u0621-\u063A\u0641-\u064A]+$")

def _setup_farasa():
    if shutil.which("java") is None:
        subprocess.run("apt-get update -qq && apt-get install -y -qq default-jre-headless", shell=True, check=True)
    if not os.path.exists(f"{FARASA_ROOT}/FarasaSrc/dist/Farasa.jar"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/qcri/FarasaSegmenter", FARASA_ROOT], check=True)
    try:
        from arabert.preprocess import ArabertPreprocessor  # noqa
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arabert"], check=True)

def _farasa_words(words):
    """Segment Arabic words (one per line) with QCRI Farasa.jar; keep its '+' split points but restore the original
    spelling (this release also normalises letters, e.g. ة->ه). Words that cannot be aligned stay unsegmented."""
    words = [w for w in dict.fromkeys(words) if AR_WORD.match(w)]
    with tempfile.TemporaryDirectory() as td:
        inp, out = f"{td}/in.txt", f"{td}/out.txt"
        open(inp, "w", encoding="utf-8").write("\n".join(words) + "\n")
        subprocess.run(["java", "-Dfile.encoding=UTF-8", "-cp", f"{FARASA_ROOT}/FarasaSrc/dist/Farasa.jar:{FARASA_ROOT}/FarasaSrc/lib/*",
                        "farasa.TestCase", "-i", inp, "-o", out], env=dict(os.environ, FarasaDataDir=f"{FARASA_ROOT}/FarasaData/"),
                       check=True, capture_output=True, timeout=3600)
        segs = open(out, encoding="utf-8").read().split("\n")[:len(words)]
    table, ok = {}, 0
    for w, s in zip(words, segs):
        parts = s.strip().split("+")
        if " " not in s.strip() and all(parts) and len("".join(parts)) == len(w):
            pieces, i = [], 0
            for p in parts: pieces.append(w[i:i + len(p)]); i += len(p)
            table[w] = "+".join(pieces); ok += 1
        else:
            table[w] = w
    print(f"  Farasa: {ok}/{len(words)} unique Arabic words segmented ({100 * ok / max(len(words), 1):.1f}%)")
    return table

def _arabert_v1_batch(texts):
    """AraBERT v1 preprocessing: v1 cleaning (identical to v0.1's), Farasa segmentation, AraBERT's own '+' marking."""
    _setup_farasa()
    from arabert.preprocess import ArabertPreprocessor
    cleaner = ArabertPreprocessor(model_name="bert-base-arabertv01")
    assert not cleaner.keep_emojis
    cleaned = [cleaner.preprocess(str(t)) for t in texts]
    table = _farasa_words(w for c in cleaned for w in c.split())
    seg_lines = {c: " ".join(table.get(w, w) for w in c.split()) for c in cleaned}
    class _Lookup:
        def segment(self, line): return seg_lines.get(line, line)
    cleaner.farasa_segmenter = _Lookup()
    return [" ".join(cleaner._tokenize_arabic_words_farasa(c).split()) for c in cleaned]

if "_old_get_text_fn" not in globals(): _old_get_text_fn = get_text_fn
def get_text_fn(spec):
    if spec == "arabert:bert-base-arabert":
        fn = lambda t: _arabert_v1_batch([t])[0]
        fn.batch = _arabert_v1_batch
        return fn
    return _old_get_text_fn(spec)

def _apply(fn, texts):
    texts = list(texts)
    return fn.batch(texts) if hasattr(fn, "batch") else [fn(t) for t in texts]

def _read(path, dataset, text_fn):
    df = pd.read_csv(path)
    raw = df[LABEL_COL].astype(str).str.strip().str.lower().map(LABEL_ALIASES)
    assert raw.notna().all(), f"{path}: unknown labels"
    out = pd.DataFrame({"id": df["id"] if "id" in df.columns else np.arange(len(df)), "label": raw})
    out["y"] = out["label"].map({l: i for i, l in enumerate(DATASETS[dataset]["labels"])})
    out["text"] = _apply(text_fn, df[TEXT_COL])
    return out

# quick check before the full run
print(_arabert_v1_batch(["والله الخدمة في هذا المطعم ممتازة جدا @user http://x.co", "والكتاب الجميل بأصواتهم #رمضان"]))

## 18. Linguistic challenge set: agreement, adjudication, validated set, evaluation
**Files** (in `marbert_repro/challenge/`): `challenge_key.csv` (coordinator only), the two returned workbooks
`Annotator_A.xlsx` and `Annotator_B.xlsx`, and, after step 18a, `adjudication.xlsx`.

- **18a** reads both workbooks, reports agreement (Cohen's kappa), and writes `adjudication.xlsx` listing only the pairs where the annotators' decisions differ. The adjudicator fills the yellow `final_*` columns.
- **18b** builds `challenge_validated.csv`: a pair is kept if the final variant sentiment equals the source label **and** meaning is preserved **and** the rewrite is natural **and** the transformation was applied. Rejected pairs are counted by reason.
- **18c** trains MARBERT and AraBERT (dev-selected, seed 42) while scoring the challenge set every epoch.
- **18d** reports, per model and transformation: macro-F1 (over positive/negative) on originals vs variants, the drop, pair consistency, flips and McNemar's test; plus a supporting analysis by the existing ArSarcasm sarcasm labels.

In [ ]:
from sklearn.metrics import cohen_kappa_score
def _read_annotator(path):
    p1 = pd.read_excel(path, sheet_name="Part1_Sentiment").query("item_id != 'EXAMPLE'")
    p2 = pd.read_excel(path, sheet_name="Part2_Pairs").query("pair_id != 'EXAMPLE'")
    p1["sentiment"] = p1["sentiment"].astype(str).str.strip().str.lower()
    for c in ["meaning_preserved", "natural", "transformation_applied"]: p2[c] = p2[c].astype(str).str.strip().str.lower()
    p1 = p1.rename(columns={"note": "note_p1"}); p2 = p2.rename(columns={"note": "note_p2"})
    return p1[["item_id", "sentiment", "note_p1"]], p2[["pair_id", "meaning_preserved", "natural", "transformation_applied", "note_p2"]]

def _decision(df, s):
    ok = (df[f"sentiment_{s}"] == df["gold"]) & (df[f"meaning_preserved_{s}"] == "yes") & (df[f"natural_{s}"] == "yes") & (df[f"transformation_applied_{s}"] == "yes")
    return ok

ann_key = pd.read_csv(f"{CHALLENGE_DIR}/challenge_key.csv")
ann_paths = {s: f"{CHALLENGE_DIR}/Annotator_{s}.xlsx" for s in "AB"}
if all(os.path.exists(p) for p in ann_paths.values()):
    ann = ann_key.copy()
    for s, p in ann_paths.items():
        p1, p2 = _read_annotator(p)
        ann = ann.merge(p1.add_suffix(f"_{s}").rename(columns={f"item_id_{s}": "item_id"}), on="item_id", how="left")
        ann = ann.merge(p2.add_suffix(f"_{s}").rename(columns={f"pair_id_{s}": "pair_id"}), on="pair_id", how="left")
        ann[f"accept_{s}"] = _decision(ann, s)
    missing = ann[(ann.sentiment_A.isin(["nan", ""])) | (ann.sentiment_B.isin(["nan", ""]))]
    if len(missing): print(f"WARNING: {len(missing)} pairs have an empty sentiment from at least one annotator")
    agree = pd.DataFrame([
        dict(judgement="variant sentiment (4 labels)", agreement=(ann.sentiment_A == ann.sentiment_B).mean(), kappa=cohen_kappa_score(ann.sentiment_A, ann.sentiment_B)),
        dict(judgement="meaning preserved", agreement=(ann.meaning_preserved_A == ann.meaning_preserved_B).mean(), kappa=cohen_kappa_score(ann.meaning_preserved_A, ann.meaning_preserved_B)),
        dict(judgement="natural", agreement=(ann.natural_A == ann.natural_B).mean(), kappa=cohen_kappa_score(ann.natural_A, ann.natural_B)),
        dict(judgement="transformation applied", agreement=(ann.transformation_applied_A == ann.transformation_applied_B).mean(), kappa=cohen_kappa_score(ann.transformation_applied_A, ann.transformation_applied_B)),
        dict(judgement="overall accept / reject", agreement=(ann.accept_A == ann.accept_B).mean(), kappa=cohen_kappa_score(ann.accept_A, ann.accept_B)),
    ]).round(3)
    agree.to_csv(f"{CHALLENGE_DIR}/agreement.csv", index=False)
    display(agree) if IN_COLAB else print(agree.to_string())

    # 18a: adjudication file for disagreements only (not overwritten once it exists)
    adj_path = f"{CHALLENGE_DIR}/adjudication.xlsx"
    dis = ann[ann.accept_A != ann.accept_B]
    print(f"{len(dis)} of {len(ann)} pairs need adjudication")
    if not os.path.exists(adj_path) and len(dis):
        cols = ["pair_id", "transformation", "gold", "source_text", "variant_text",
                "sentiment_A", "sentiment_B", "meaning_preserved_A", "meaning_preserved_B", "natural_A", "natural_B",
                "transformation_applied_A", "transformation_applied_B", "note_p1_A", "note_p1_B", "note_p2_A", "note_p2_B"]
        out = dis[cols].copy()
        for c in ["final_sentiment", "final_meaning_preserved", "final_natural", "final_transformation_applied", "adjudicator_note"]: out[c] = ""
        out.to_excel(adj_path, index=False)
        from openpyxl import load_workbook
        from openpyxl.styles import PatternFill, Font
        from openpyxl.worksheet.datavalidation import DataValidation
        wb = load_workbook(adj_path); ws = wb.active
        n = ws.max_row
        for col in range(18, 23):
            for r in range(2, n + 1): ws.cell(row=r, column=col).fill = PatternFill("solid", start_color="FFF2CC")
        for row in ws.iter_rows():
            for c in row: c.font = Font(name="Arial", bold=(c.row == 1))
        dv1 = DataValidation(type="list", formula1='"positive,negative,neutral,unclear"'); ws.add_data_validation(dv1); dv1.add(f"R2:R{n}")
        dv2 = DataValidation(type="list", formula1='"yes,no"'); ws.add_data_validation(dv2); dv2.add(f"S2:U{n}")
        wb.save(adj_path)
        print("Wrote", adj_path, "-> fill the yellow final_* columns, upload it back, then run this cell again")

    # 18b: final decision and validated set
    ann["final_accept"] = ann.accept_A & ann.accept_B
    if os.path.exists(adj_path):
        adj = pd.read_excel(adj_path).fillna("")
        filled = adj[adj["final_sentiment"].astype(str).str.len() > 0]
        if len(filled) < len(adj): print(f"NOTE: {len(adj) - len(filled)} adjudications still empty - those pairs are rejected for now")
        for _, a in filled.iterrows():
            ok = (str(a.final_sentiment).lower() == a.gold) and all(str(a[c]).lower() == "yes" for c in ["final_meaning_preserved", "final_natural", "final_transformation_applied"])
            ann.loc[ann.pair_id == a.pair_id, "final_accept"] = ok
    def reason(r):
        if r.final_accept: return "kept"
        if r.accept_A or r.accept_B: return "rejected after adjudication"
        rs = []
        if r.sentiment_A != r.gold and r.sentiment_B != r.gold: rs.append("sentiment changed or unclear")
        if r.meaning_preserved_A == "no" and r.meaning_preserved_B == "no": rs.append("meaning not preserved")
        if r.natural_A == "no" and r.natural_B == "no": rs.append("not natural")
        if r.transformation_applied_A == "no" and r.transformation_applied_B == "no": rs.append("transformation not applied")
        return "; ".join(rs) or "rejected by both (different reasons)"
    ann["outcome"] = ann.apply(reason, axis=1)
    ann.to_csv(f"{CHALLENGE_DIR}/annotation_merged.csv", index=False)
    summary = ann.groupby(["transformation", "outcome"]).size().unstack(fill_value=0)
    display(summary) if IN_COLAB else print(summary.to_string())
    validated = ann[ann.final_accept][["pair_id", "transformation", "source_id", "gold", "source_sarcasm", "source_text", "variant_text"]]
    validated.to_csv(f"{CHALLENGE_DIR}/challenge_validated.csv", index=False)
    print(f"Validated challenge set: {len(validated)} pairs ->", f"{CHALLENGE_DIR}/challenge_validated.csv")
else:
    print("Annotator workbooks not found in", CHALLENGE_DIR)

In [ ]:
# 18c: train both models (dev-selected, seed 42) and score the validated challenge set every epoch
RUN_CHALLENGE = True
if RUN_CHALLENGE and os.path.exists(f"{CHALLENGE_DIR}/challenge_validated.csv"):
    for mk in MODELS:
        train_one_run_extra("challenge", "arsarcasm", mk, 42, protocol="independent", challenge=True)

In [ ]:
# 18d: challenge-set results at the dev-selected epoch + sarcasm-label support analysis
def _mcnemar_exact(b, c):
    n, k = b + c, min(b, c)
    if n == 0: return 1.0
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)

def challenge_results():
    labels = DATASETS["arsarcasm"]["labels"]; pos_neg = [labels.index("positive"), labels.index("negative")]
    rows, flips = [], []
    for mk in MODELS:
        rd = f"{OUT_DIR_EXTRA}/challenge/arsarcasm/{mk}/seed42"
        if not os.path.exists(f"{rd}/DONE"): continue
        ep = [json.loads(l) for l in open(f"{rd}/metrics_per_epoch.jsonl")]
        e = max(ep, key=lambda r: (r["dev"]["f1_pn"], -r["epoch"]))["epoch"]           # chosen on dev
        o = pd.read_csv(f"{rd}/preds/epoch_{e:02d}_ch_orig_preds.csv"); v = pd.read_csv(f"{rd}/preds/epoch_{e:02d}_ch_var_preds.csv")
        m = o.merge(v, on=["pair_id", "gold"], suffixes=("_orig", "_var")).merge(pd.read_csv(f"{CHALLENGE_DIR}/challenge_validated.csv")[["pair_id", "transformation"]], on="pair_id")
        for tf, g in list(m.groupby("transformation")) + [("all", m)]:
            f_o = 100 * f1_score(g.gold, g.pred_orig, labels=pos_neg, average="macro", zero_division=0)
            f_v = 100 * f1_score(g.gold, g.pred_var, labels=pos_neg, average="macro", zero_division=0)
            co, cv = g.pred_orig == g.gold, g.pred_var == g.gold
            b, c = int((co & ~cv).sum()), int((~co & cv).sum())
            rows.append(dict(model=mk, epoch=e, transformation=tf, pairs=len(g),
                             macro_f1_original=round(f_o, 2), macro_f1_variant=round(f_v, 2), drop=round(f_o - f_v, 2),
                             pair_consistency=round(100 * (g.pred_orig == g.pred_var).mean(), 1),
                             correct_to_wrong=b, wrong_to_correct=c, mcnemar_p=round(_mcnemar_exact(b, c), 4)))
        m["model"] = mk; flips.append(m[(m.pred_orig == m.gold) & (m.pred_var != m.gold)])
    res = pd.DataFrame(rows)
    if res.empty: print("No finished challenge runs yet."); return None
    res.to_csv(f"{CHALLENGE_DIR}/challenge_results.csv", index=False)
    display(res) if IN_COLAB else print(res.to_string())
    fl = pd.concat(flips).merge(pd.read_csv(f"{CHALLENGE_DIR}/annotation_merged.csv")[["pair_id", "source_text", "variant_text", "sentiment_A", "sentiment_B", "note_p1_A", "note_p1_B", "note_p2_A", "note_p2_B"]], on="pair_id")
    fl.to_csv(f"{CHALLENGE_DIR}/challenge_flips_for_inspection.csv", index=False)
    print(f"{len(fl)} correct->wrong flips written for inspection (linguistic weakness vs annotation ambiguity)")
    return res

def sarcasm_support():
    """Existing ArSarcasm sarcasm labels: F1-PN on sarcastic vs non-sarcastic test tweets (dev-selected runs, all seeds)."""
    test = pd.read_csv(f"{DEV_DATA_DIR}/arsarcasm_test.csv")[["id", "sarcasm"]]
    labels = DATASETS["arsarcasm"]["labels"]; pos_neg = [labels.index("positive"), labels.index("negative")]
    rows = []
    for mk in MODELS:
        for s in SEEDS:
            rd = f"{OUT_DIR_DEV}/arsarcasm/{mk}/seed{s}"
            if not os.path.exists(f"{rd}/DONE"): continue
            ep = [json.loads(l) for l in open(f"{rd}/metrics_per_epoch.jsonl")]
            e = max(ep, key=lambda r: (r["dev"]["f1_pn"], -r["epoch"]))["epoch"]
            p = pd.read_csv(f"{rd}/preds/epoch_{e:02d}_test_preds.csv").merge(test, on="id")
            for sar, g in p.groupby("sarcasm"):
                rows.append(dict(model=mk, seed=s, sarcastic=bool(sar), tweets=len(g),
                                 f1_pn=100 * f1_score(g.gold, g.pred, labels=pos_neg, average="macro", zero_division=0)))
    df = pd.DataFrame(rows)
    if df.empty: print("No finished dev-selected ArSarcasm runs yet."); return None
    out = df.groupby(["model", "sarcastic"]).agg(tweets=("tweets", "first"), f1_pn_mean=("f1_pn", "mean"), f1_pn_sd=("f1_pn", "std")).round(2)
    out.to_csv(f"{CHALLENGE_DIR}/sarcasm_support.csv")
    display(out) if IN_COLAB else print(out.to_string())
    return out

ch_res = challenge_results() if os.path.exists(f"{CHALLENGE_DIR}/challenge_validated.csv") else print("Run 18a-18c first.")
sar = sarcasm_support()

## 19. Export package for Member 3 (controlled evaluation)
Needs no GPU: run sections 1–6, 11 and 12 (set the training flags to `False`), then this section.
Writes `outputs/member3_package/` and a zip of it, with:

| File | Content |
|---|---|
| `epoch_metrics_long.csv` | Every epoch of every run, both protocols: F1-PN, macro-F1, accuracy, per-class F1 (dev and test for the independent protocol) |
| `checkpoint_diagnostic.csv` | Per run: best-on-test epoch and score, dev-selected epoch and its test score, last-epoch (25) score, and the gaps between them |
| `seed_variation.csv` | Per protocol, dataset and model: mean, SD, min, max, range over seeds of the reported score |
| `f1pn_vs_macrof1.csv` | Reported-epoch F1-PN, macro-F1 and accuracy per run, and whether MARBERT > AraBERT under each metric |
| `bootstrap_ci.csv` | Paired bootstrap (1,000 resamples of test tweets) of MARBERT − AraBERT, per seed and for the mean of the 3 seeds, for the dataset metric and macro-F1 |
| copies | `reproduction_summary.csv`, `per_run_best_epoch.csv`, `independent_summary.csv`, `per_run_dev_selected.csv`, `error_analysis_predictions.csv`, `departures.md` |

**Reported epoch**: historical = best epoch on test; independent = best epoch on dev (earliest on ties).

In [ ]:
PKG = f"{PROJECT_DIR}/outputs/member3_package"
os.makedirs(PKG, exist_ok=True)
PROTOCOLS = {"historical": OUT_DIR, "independent": OUT_DIR_DEV}

def _epochs(protocol, d, mk, s):
    f = f"{PROTOCOLS[protocol]}/{d}/{mk}/seed{s}/metrics_per_epoch.jsonl"
    if not os.path.exists(f"{os.path.dirname(f)}/DONE"): return None
    return [json.loads(l) for l in open(f)]

def _test(rec, protocol):  return rec if protocol == "historical" else rec["test"]

def _reported_epoch(ep, protocol, key):
    if protocol == "historical": return max(ep, key=lambda r: (r[key], -r["epoch"]))["epoch"]
    return max(ep, key=lambda r: (r["dev"][key], -r["epoch"]))["epoch"]

long_rows, diag_rows, rep_rows = [], [], []
for protocol in PROTOCOLS:
    for d in DATASETS:
        key, labels = DATASETS[d]["metric"], DATASETS[d]["labels"]
        for mk in MODELS:
            for s in SEEDS:
                ep = _epochs(protocol, d, mk, s)
                if ep is None: continue
                for r in ep:
                    for split in (["test"] if protocol == "historical" else ["dev", "test"]):
                        m = r if protocol == "historical" else r[split]
                        row = dict(protocol=protocol, dataset=d, model=mk, seed=s, epoch=r["epoch"], split=split,
                                   train_loss=r["train_loss"], f1_pn=m.get("f1_pn"), macro_f1=m["macro_f1"], accuracy=m["accuracy"])
                        for l in labels: row[f"f1_{l}"] = m["per_class"][l]["f1"]
                        long_rows.append(row)
                best_test = max(ep, key=lambda r: (_test(r, protocol)[key], -r["epoch"]))
                last = ep[-1]
                rep_e = _reported_epoch(ep, protocol, key)
                rep = next(r for r in ep if r["epoch"] == rep_e)
                diag = dict(protocol=protocol, dataset=d, model=mk, seed=s, metric=key,
                            best_test_epoch=best_test["epoch"], best_test_score=_test(best_test, protocol)[key],
                            last_epoch=last["epoch"], last_epoch_score=_test(last, protocol)[key])
                diag["best_minus_last"] = diag["best_test_score"] - diag["last_epoch_score"]
                if protocol == "independent":
                    diag.update(dev_selected_epoch=rep_e, test_at_dev_epoch=rep["test"][key],
                                selection_optimism=diag["best_test_score"] - rep["test"][key],
                                same_epoch=rep_e == best_test["epoch"])
                diag_rows.append(diag)
                t = _test(rep, protocol)
                rep_rows.append(dict(protocol=protocol, dataset=d, model=mk, seed=s, reported_epoch=rep_e,
                                     score=t[key], f1_pn=t.get("f1_pn"), macro_f1=t["macro_f1"], accuracy=t["accuracy"]))

long_df, diag_df, rep_df = pd.DataFrame(long_rows), pd.DataFrame(diag_rows).round(3), pd.DataFrame(rep_rows)
long_df.round(4).to_csv(f"{PKG}/epoch_metrics_long.csv", index=False)
diag_df.to_csv(f"{PKG}/checkpoint_diagnostic.csv", index=False)

# seed variation of the reported score
seed_df = (rep_df.groupby(["protocol", "dataset", "model"])["score"]
           .agg(n_seeds="count", mean="mean", sd="std", min="min", max="max").reset_index())
seed_df["range"] = seed_df["max"] - seed_df["min"]
seed_df.round(3).to_csv(f"{PKG}/seed_variation.csv", index=False)

# F1-PN vs macro-F1 (and accuracy): means per model and whether the ranking holds under each metric
fm = rep_df.groupby(["protocol", "dataset", "model"])[["f1_pn", "macro_f1", "accuracy"]].mean().unstack("model")
out = []
for (protocol, d), r in fm.iterrows():
    row = dict(protocol=protocol, dataset=d)
    for met in ["f1_pn", "macro_f1", "accuracy"]:
        mar, ara = r[(met, "marbert")], r[(met, "arabert")]
        row.update({f"{met}_marbert": mar, f"{met}_arabert": ara, f"{met}_lead": mar - ara, f"{met}_marbert_wins": bool(mar > ara)})
    out.append(row)
f1cmp = pd.DataFrame(out).round(3)
f1cmp.to_csv(f"{PKG}/f1pn_vs_macrof1.csv", index=False)

# paired bootstrap of MARBERT - AraBERT on the same test tweets
def _f1_counts(gold, pred, idx, cls):
    g, p = gold[idx], pred[idx]
    tp = ((g == cls) & (p == cls)).sum(1); fp = ((g != cls) & (p == cls)).sum(1); fn = ((g == cls) & (p != cls)).sum(1)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)

def _metric(gold, pred, idx, metric, labels):
    if metric == "accuracy": return 100 * (gold[idx] == pred[idx]).mean(1)
    classes = [labels.index("positive"), labels.index("negative")] if metric == "f1_pn" else list(range(len(labels)))
    return 100 * np.mean([_f1_counts(gold, pred, idx, c) for c in classes], axis=0)

def _preds(protocol, d, mk, s, epoch):
    p = pd.read_csv(f"{PROTOCOLS[protocol]}/{d}/{mk}/seed{s}/preds/epoch_{epoch:02d}_test_preds.csv")
    return p["gold"].values, p["pred"].values

B, rng = 1000, np.random.default_rng(12345)
boot_rows = []
for protocol in PROTOCOLS:
    for d in DATASETS:
        labels = DATASETS[d]["labels"]
        seeds_done = [s for s in SEEDS if all(((rep_df.protocol == protocol) & (rep_df.dataset == d) & (rep_df.model == mk) & (rep_df.seed == s)).any() for mk in MODELS)]
        if not seeds_done: continue
        n = len(_preds(protocol, d, "marbert", seeds_done[0], int(rep_df[(rep_df.protocol == protocol) & (rep_df.dataset == d) & (rep_df.model == "marbert") & (rep_df.seed == seeds_done[0])].reported_epoch.iloc[0]))[0])
        idx = rng.integers(0, n, size=(B, n))                     # same resampled tweets for both models and all seeds
        for metric in dict.fromkeys([DATASETS[d]["metric"], "macro_f1"]):
            diffs = []
            for s in seeds_done:
                vals = {}
                for mk in MODELS:
                    e = int(rep_df[(rep_df.protocol == protocol) & (rep_df.dataset == d) & (rep_df.model == mk) & (rep_df.seed == s)].reported_epoch.iloc[0])
                    gold, pred = _preds(protocol, d, mk, s, e)
                    vals[mk] = (gold, pred)
                assert (vals["marbert"][0] == vals["arabert"][0]).all(), "test order differs between models"
                full = np.arange(n)[None, :]
                obs = _metric(*vals["marbert"], full, metric, labels)[0] - _metric(*vals["arabert"], full, metric, labels)[0]
                bd = _metric(*vals["marbert"], idx, metric, labels) - _metric(*vals["arabert"], idx, metric, labels)
                diffs.append(bd)
                lo, hi = np.percentile(bd, [2.5, 97.5])
                boot_rows.append(dict(protocol=protocol, dataset=d, metric=metric, seed=str(s), observed_diff=obs,
                                      ci_low=lo, ci_high=hi, p_marbert_not_better=(bd <= 0).mean(), excludes_zero=bool(lo > 0 or hi < 0)))
            md_ = np.mean(diffs, axis=0)
            lo, hi = np.percentile(md_, [2.5, 97.5])
            obs_mean = np.mean([r["observed_diff"] for r in boot_rows if r["protocol"] == protocol and r["dataset"] == d and r["metric"] == metric and r["seed"] != "mean"])
            boot_rows.append(dict(protocol=protocol, dataset=d, metric=metric, seed="mean", observed_diff=obs_mean,
                                  ci_low=lo, ci_high=hi, p_marbert_not_better=(md_ <= 0).mean(), excludes_zero=bool(lo > 0 or hi < 0)))
boot_df = pd.DataFrame(boot_rows).round(3)
boot_df.to_csv(f"{PKG}/bootstrap_ci.csv", index=False)

# copies of the summary files
for src in [f"{OUT_DIR}/reproduction_summary.csv", f"{OUT_DIR}/per_run_best_epoch.csv",
            f"{OUT_DIR_DEV}/independent_summary.csv", f"{OUT_DIR_DEV}/per_run_dev_selected.csv",
            f"{PROJECT_DIR}/outputs/error_analysis_predictions.csv", f"{PROJECT_DIR}/departures.md"]:
    if os.path.exists(src): shutil.copy(src, PKG)
    else: print("not found (run its section first):", src)
zip_path = shutil.make_archive(f"{PROJECT_DIR}/outputs/member3_package", "zip", PKG)
print("Package:", PKG, "\nZip:", zip_path)
for name, df in [("checkpoint diagnostic (means)", diag_df.groupby(["protocol", "dataset", "model"])[[c for c in ["best_minus_last", "selection_optimism"] if c in diag_df]].mean().round(2)),
                 ("seed variation", seed_df.round(2)), ("F1-PN vs macro-F1", f1cmp), ("paired bootstrap, mean of seeds", boot_df[boot_df.seed == "mean"])]:
    print(f"\n== {name}"); display(df) if IN_COLAB else print(df.to_string())

Package: /content/drive/MyDrive/marbert_repro/outputs/member3_package 
Zip: /content/drive/MyDrive/marbert_repro/outputs/member3_package.zip

== checkpoint diagnostic (means)


best_minus_last  selection_optimism
protocol    dataset   model                                       
historical  ajgt      arabert             1.11                 NaN
                      marbert             0.83                 NaN
            arsarcasm arabert             1.96                 NaN
                      marbert             1.64                 NaN
            astd      arabert             1.55                 NaN
                      marbert             1.33                 NaN
independent ajgt      arabert             0.83                0.56
                      marbert             0.28                0.74
            arsarcasm arabert             2.15                0.98
                      marbert             1.48                1.67
            astd      arabert             1.25                0.25
                      marbert             0.55                0.65


== seed variation


,protocol,dataset,model,n_seeds,mean,sd,min,max,range
0,historical,ajgt,arabert,3,88.89,0.73,88.06,89.44,1.39
1,historical,ajgt,marbert,3,94.35,0.32,94.17,94.72,0.56
2,historical,arsarcasm,arabert,3,67.86,0.31,67.63,68.22,0.59
3,historical,arsarcasm,marbert,3,72.42,0.66,71.97,73.18,1.20
4,historical,astd,arabert,3,75.83,0.17,75.68,76.01,0.33
5,historical,astd,marbert,3,80.90,0.55,80.48,81.52,1.03
6,independent,ajgt,arabert,3,88.06,1.11,86.94,89.17,2.22
7,independent,ajgt,marbert,3,93.52,0.32,93.33,93.89,0.56
8,independent,arsarcasm,arabert,3,66.75,0.68,66.17,67.50,1.33
9,independent,arsarcasm,marbert,3,70.80,0.28,70.52,71.07,0.55



== F1-PN vs macro-F1


,protocol,dataset,f1_pn_marbert,f1_pn_arabert,f1_pn_lead,f1_pn_marbert_wins,macro_f1_marbert,macro_f1_arabert,macro_f1_lead,macro_f1_marbert_wins,accuracy_marbert,accuracy_arabert,accuracy_lead,accuracy_marbert_wins
0,historical,ajgt,94.348,88.881,5.467,True,94.348,88.881,5.467,True,94.352,88.889,5.463,True
1,historical,arsarcasm,72.421,67.862,4.559,True,74.814,71.291,3.523,True,76.983,74.392,2.591,True
2,historical,astd,80.896,75.831,5.065,True,68.248,61.539,6.709,True,72.617,67.574,5.043,True
3,independent,ajgt,93.516,88.055,5.460,True,93.516,88.055,5.460,True,93.519,88.056,5.463,True
4,independent,arsarcasm,70.796,66.754,4.043,True,73.655,70.536,3.120,True,76.319,73.870,2.449,True
5,independent,astd,79.850,75.546,4.305,True,64.492,61.611,2.881,True,70.751,67.675,3.076,True



== paired bootstrap, mean of seeds


,protocol,dataset,metric,seed,observed_diff,ci_low,ci_high,p_marbert_not_better,excludes_zero
3,historical,arsarcasm,f1_pn,mean,4.559,2.662,6.555,0.000,True
7,historical,arsarcasm,macro_f1,mean,3.523,2.002,5.134,0.000,True
11,historical,astd,f1_pn,mean,5.065,2.691,7.580,0.000,True
15,historical,astd,macro_f1,mean,6.709,3.342,9.857,0.000,True
19,historical,ajgt,accuracy,mean,5.463,3.056,8.148,0.000,True
23,historical,ajgt,macro_f1,mean,5.467,3.054,8.192,0.000,True
27,independent,arsarcasm,f1_pn,mean,4.043,2.143,6.067,0.000,True
31,independent,arsarcasm,macro_f1,mean,3.120,1.516,4.802,0.000,True
35,independent,astd,f1_pn,mean,4.305,1.979,6.631,0.000,True
39,independent,astd,macro_f1,mean,2.881,-0.213,5.817,0.029,False


# 20. Export Prediction with agjt ucb

In [11]:
# Add the AJGT authors'-split runs (ajgt_ubc) to the error-analysis export for Member 1.
# Separate cell: run after sections 1-6. Retrains nothing; reads the finished ajgt_ubc runs.
# Keeps every existing row of error_analysis_predictions.csv and only adds or replaces the ajgt_ubc rows.
import os, re, json, subprocess, pandas as pd

EA_PATH  = f"{PROJECT_DIR}/outputs/error_analysis_predictions.csv"
KEY_PATH = f"{PROJECT_DIR}/outputs/ajgt_ubc_test_key.csv"

# 1. Same setup as the ajgt_ubc training cell (safe to re-run)
if "ajgt_ubc" not in DATASETS:
    DATASETS["ajgt_ubc"] = dict(DATASETS["ajgt"])
if not os.path.exists(f"{DATA_DIR}/ajgt_ubc/test.csv"):
    src = "/content/marbert_repo"
    if not os.path.exists(src):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/UBC-NLP/marbert", src], check=True)
    os.makedirs(f"{DATA_DIR}/ajgt_ubc", exist_ok=True)
    for sp in ["train", "test"]:
        d = pd.read_csv(f"{src}/examples/UBC_AJGT_final_shuffled_{sp}.tsv", sep="\t", dtype=str, quoting=3)
        d.rename(columns={"content": "text"})[["text", "label"]].to_csv(f"{DATA_DIR}/ajgt_ubc/{sp}.csv", index=False)

# 2. Key file: one id per test tweet of the authors' split, its text, gold label,
#    and Member 1's AJGT id when the same tweet is also in our AJGT test set
def _norm(t):
    t = re.sub(r"[إأآا]", "ا", str(t)); t = t.replace("ة", "ه").replace("ى", "ي")
    return re.sub(r"[^ء-ي]", "", t)          # Arabic letters only (the authors' files are preprocessed)

ubc_test = load_split("ajgt_ubc", "test")              # same rows, same order as training used
raw_text = pd.read_csv(f"{DATA_DIR}/ajgt_ubc/test.csv")["text"].astype(str)
key = pd.DataFrame({"id": [f"ajgt_ubc_test_{i:05d}" for i in range(len(ubc_test))],
                    "text": raw_text.values, "gold": ubc_test["label"].values})
m1_path = f"{DEV_DATA_DIR}/ajgt_test.csv"
if os.path.exists(m1_path):
    m1 = pd.read_csv(m1_path)
    lookup = dict(zip(m1["text"].map(_norm), m1["id"]))
    key["m1_ajgt_id"] = key["text"].map(_norm).map(lookup)
    print(f"Tweets also in our AJGT test set: {key['m1_ajgt_id'].notna().sum()} of {len(key)}  (expect about 47)")
key.to_csv(KEY_PATH, index=False)

# 3. Predictions at the reported epoch (historical protocol: best epoch on test), same columns as section 15
labels, metric = DATASETS["ajgt_ubc"]["labels"], DATASETS["ajgt_ubc"]["metric"]
frames = []
for mk in MODELS:
    for s in SEEDS:
        rd = f"{OUT_DIR}/ajgt_ubc/{mk}/seed{s}"
        if not os.path.exists(f"{rd}/DONE"):
            print("missing run:", rd); continue
        ep = [json.loads(l) for l in open(f"{rd}/metrics_per_epoch.jsonl")]
        best = max(ep, key=lambda e: (e[metric], -e["epoch"]))["epoch"]
        p = pd.read_csv(f"{rd}/preds/epoch_{best:02d}_test_preds.csv")
        out = pd.DataFrame({"dataset": "ajgt_ubc", "model": mk, "protocol": "historical", "seed": s, "epoch": best,
                            "id": key["id"].values[p["idx"].values],
                            "gold": [labels[g] for g in p["gold"]], "pred": [labels[q] for q in p["pred"]]})
        assert (out["gold"].values == key["gold"].values[p["idx"].values]).all(), "gold labels do not match the test file"
        out["correct"] = out["gold"] == out["pred"]
        for l in labels: out[f"p_{l}"] = p[f"p_{l}"].values
        frames.append(out)

# 4. Merge into the existing export (drops any earlier ajgt_ubc rows, so re-running is safe)
new = pd.concat(frames, ignore_index=True)
old = pd.read_csv(EA_PATH) if os.path.exists(EA_PATH) else pd.DataFrame()
if old.empty: print("NOTE: no existing export found - run section 15 first to include the other datasets.")
else: old = old[old["dataset"] != "ajgt_ubc"]
ea = pd.concat([old, new], ignore_index=True)
ea.to_csv(EA_PATH, index=False)
print(f"Wrote {len(ea):,} rows ({len(new):,} ajgt_ubc) -> {EA_PATH}\nKey file -> {KEY_PATH}")
print(ea.groupby(["protocol", "dataset", "model"]).agg(runs=("seed", "nunique"), accuracy=("correct", "mean")).round(4))

Tweets also in our AJGT test set: 47 of 360  (expect about 47)
Wrote 39,732 rows (2,160 ajgt_ubc) -> /content/drive/MyDrive/marbert_repro/outputs/error_analysis_predictions.csv
Key file -> /content/drive/MyDrive/marbert_repro/outputs/ajgt_ubc_test_key.csv
                               runs  accuracy
protocol    dataset   model                  
historical  ajgt      arabert     3    0.8889
                      marbert     3    0.9435
            ajgt_ubc  arabert     3    0.9213
                      marbert     3    0.9583
            arsarcasm arabert     3    0.7439
                      marbert     3    0.7698
            astd      arabert     3    0.6757
                      marbert     3    0.7262
independent ajgt      arabert     3    0.8806
                      marbert     3    0.9352
            arsarcasm arabert     3    0.7387
                      marbert     3    0.7632
            astd      arabert     3    0.6768
                      marbert     3    0.7075
